# 🏠 Housekeeper — LTB Evidence Brief Generator

A tool that helps you create professional evidence briefs for **Ontario Landlord and Tenant Board (LTB)** hearings.

Upload your evidence files — photos, PDFs, Word documents, spreadsheets, even iPhone HEIC images — organize them into numbered tabs, and generate a court-ready PDF complete with a title page, table of contents, tab dividers, and properly numbered pages.

---

## 🚀 How to Use This Notebook

This notebook launches a local web app inside Google Colab using a public tunnel (ngrok). Follow the steps below:

### Before You Start
- You need a free **ngrok** account to expose the app publicly. Sign up at [ngrok.com](https://ngrok.com) and copy your **authtoken** from the dashboard.
- When prompted in **Step 8**, paste your ngrok authtoken to get a public URL for the app.

### Steps

| Step | What it Does |
|------|--------------|
| **Step 1** | Installs required Python packages (Flask, ReportLab, Pillow, PyMuPDF, etc.) |
| **Step 2** | Creates the `/content/uploads` and `/content/templates` working directories |
| **Steps 3–5** *(Optional)* | Upload and analyze past LTB decisions to generate evidence frequency statistics — skip these if you just want to use the tool |
| **Step 6** | Writes the full web frontend (`index.html`) to disk |
| **Step 7** | Writes the Flask backend (`app.py`) to disk |
| **Step 8** | Starts the Flask server and opens a public tunnel — **click the link that appears to open the app** |

### Using the Web App (4 Steps)

1. **Case Information** — Enter your LTB file number, hearing date, party names, and case type (N4/N5).
2. **Upload Files** — Drag and drop your evidence files (photos, PDFs, Word docs, spreadsheets, etc.).
3. **Arrange Tabs** — Organize your files into named evidence tabs (e.g., *Lease Agreement*, *Rent Payment Records*).
4. **Review & Generate** — Redact any sensitive information if needed, then click **Generate PDF** to download your court-ready Evidence Brief.

### Tips
- The app runs entirely in your Colab session — no data is stored externally.
- Use the suggested tab names (shown after selecting your case type) to match common evidence categories adjudicators look for.
- Refer to your evidence by **page number** during the hearing — the Table of Contents in your PDF shows each tab's starting page.
- Submit your completed brief through the **Tribunals Ontario Portal** and serve a copy to the opposing party before your hearing deadline.

---

*Built by Fukun Yang & Angel Xing.*


## Step 1: Install Dependencies

Install all Python packages and system tools required to run the app.

- **Flask** — web server framework  
- **Pillow / pillow-heif** — image processing and HEIC support  
- **ReportLab** — PDF creation and layout  
- **PyMuPDF / pdfplumber** — reading and converting PDFs  
- **python-docx / openpyxl** — parsing Word and Excel files  
- **sentence-transformers** *(optional)* — ML model for evidence analysis  
- **LibreOffice** — converts Word/Excel/PowerPoint files to PDF images

Run this cell once at the start of each Colab session.

In [ ]:
# Install Python packages
!pip install -q flask Pillow pillow-heif sentence-transformers reportlab PyMuPDF pdfplumber python-docx openpyxl pyngrok

# Install LibreOffice for DOCX/XLSX/PPTX → PDF conversion
!apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq libreoffice-core libreoffice-common > /dev/null 2>&1 && echo 'LibreOffice installed.' || echo 'LibreOffice install skipped (office doc conversion will be unavailable).'

print('\nAll dependencies installed.')


## Step 2: Set Up Working Directories

Creates the folder structure Flask needs to operate:

- `/content/templates/` — stores the HTML frontend (`index.html`)  
- `/content/uploads/` — stores uploaded files and generated PDFs

These directories are created if they don't already exist.

In [ ]:
import os
os.makedirs('/content/templates', exist_ok=True)
os.makedirs('/content/uploads', exist_ok=True)
print('Directories ready.')


## Evidence Extraction Pipeline (Optional)

Run the cells below to analyze a PDF of LTB N4 decisions using **local semantic analysis**.
Uses the `all-MiniLM-L6-v2` sentence-transformers model (~80MB, runs on CPU).

**All processing happens locally on this machine — no data is sent to any cloud service.**

You only need to run this once — the app ships with sensible defaults if you skip this section.
To re-run extraction, delete the existing `evidence_stats.json` first.


## Step 3 (Optional): Upload LTB Decisions PDF for Evidence Analysis

**You can skip Steps 3–5** — the app ships with sensible default evidence statistics.

Only run this if you want to regenerate the evidence checklist from your own set of LTB decisions:
1. Prepare a single PDF containing multiple LTB N4 decisions (concatenated)
2. Run this cell to upload it — it will extract all text for analysis in the next step

If `evidence_stats.json` already exists, this cell will be skipped automatically.

In [ ]:
# ─── PDF Upload + Text Extraction ───────────────
import os, pdfplumber

STATS_PATH = '/content/evidence_stats.json'

if os.path.exists(STATS_PATH):
    print('evidence_stats.json already exists — skipping upload.')
    print('Delete it first if you want to re-run extraction.')
    full_text = ''
else:
    from google.colab import files as colab_files
    print('Upload a concatenated PDF of LTB N4 decisions:')
    uploaded = colab_files.upload()
    pdf_filename = list(uploaded.keys())[0]
    print(f'Processing: {pdf_filename}')
    all_text = []
    with pdfplumber.open(pdf_filename) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if text:
                all_text.append(text)
    full_text = '\n\n'.join(all_text)
    print(f'Extracted {len(all_text)} pages, {len(full_text):,} characters total.')


## Step 4 (Optional): Evidence Analyzer Module

Writes the `evidence_analyzer.py` module to disk.

This module uses the **`all-MiniLM-L6-v2`** sentence-transformers model (~80 MB) to identify which types of evidence appear in LTB decisions using semantic similarity matching.

**Privacy note:** All analysis runs entirely on this machine — no data is sent to any external service.

Only needed if you're running the evidence extraction pipeline (Steps 3–5).

In [ ]:
%%writefile /content/evidence_analyzer.py
"""
Local evidence analyzer for LTB N4 decisions.

Uses sentence-transformers (all-MiniLM-L6-v2, ~80MB) for semantic similarity.
All processing runs locally — no data is sent to any cloud service.

Usage:
    from evidence_analyzer import analyze_pdf
    stats = analyze_pdf("decisions.pdf")
    # stats is a dict ready to write as evidence_stats.json
"""

import re
import json
import os
from collections import Counter

# ---------------------------------------------------------------------------
# Evidence categories and their semantic reference phrases
# Each category has multiple example phrases that describe what that evidence
# looks like in an LTB decision. The model compares case text against these.
#
# Phrases are chosen to be specific and distinctive — avoiding generic legal
# language that could false-positive across categories.
# ---------------------------------------------------------------------------

EVIDENCE_CATEGORIES = {
    "N4 Notice of Termination": [
        "N4 notice of termination was served",
        "the landlord served an N4 notice",
        "form N4 for non-payment of rent",
        "N4 application for termination",
        "notice of termination for non-payment of rent",
        "the N4 notice was filed with the Board",
    ],
    "Financial records (rent receipts, bank statements, rent ledger)": [
        "rent receipts were submitted as evidence",
        "bank statements showing rent deposits",
        "rent ledger documenting monthly payments",
        "financial records of rent transactions",
        "account statements from the bank were filed",
        "receipts for rent paid were entered as exhibits",
        "the landlord submitted a rent ledger as an exhibit",
    ],
    "Lease agreement": [
        "the lease agreement between the parties was submitted as evidence",
        "the tenancy agreement was entered into evidence at the hearing",
        "a signed rental agreement was filed as an exhibit",
        "the written lease document specifies the monthly rent",
        "a copy of the lease was produced by the landlord",
    ],
    "Payment history / transaction records": [
        "payment history showing rent arrears over several months",
        "e-transfer records confirming rent payments",
        "cheques provided as proof of rent payment",
        "money order receipts for rent",
        "rent arrears accumulated over several months",
        "outstanding rent owing to the landlord totals",
        "record of partial payments made by the tenant",
    ],
    "Communication records (emails, text messages, letters)": [
        "email correspondence between landlord and tenant was submitted",
        "text messages about rent were filed as exhibits",
        "demand letters sent by the landlord were entered into evidence",
        "voicemail messages were submitted as evidence",
        "written communication records were filed as exhibits",
    ],
    "Legal documents (prior orders, court filings)": [
        "a prior Board order from a previous proceeding was filed",
        "a previous LTB order regarding this tenancy was submitted",
        "a section 78 conditional order was previously granted",
        "the tenant breached a previous conditional order",
        "a previous application had been filed with the Board",
        "documents from a prior court hearing were submitted",
    ],
    "Witness testimony": [
        "a witness testified at the hearing about the tenancy",
        "oral testimony was provided by a third party witness",
        "the witness gave sworn evidence about the rental unit",
        "a sworn affidavit was submitted by a witness",
        "testimony from a third party corroborated the claim",
    ],
    "Government/third-party records (inspection reports, municipal notices)": [
        "an inspection report from the municipality was filed",
        "a by-law enforcement notice was submitted",
        "the health inspector report on the rental unit",
        "fire inspector findings were entered as evidence",
        "property standards inspection report was submitted",
        "a municipal notice regarding the property was filed",
    ],
    "Photos of unit conditions": [
        "photographs of the rental unit were submitted",
        "photos showing the condition of the unit were filed",
        "photographic evidence of the unit was entered as an exhibit",
        "pictures of damage to the property were submitted",
        "video evidence of the unit condition was shown",
    ],
    "Maintenance/repair requests or records": [
        "repair requests submitted by the tenant to the landlord",
        "maintenance records for the rental unit were filed",
        "work orders for repairs to the unit were submitted",
        "service requests for maintenance issues were entered",
        "record of maintenance performed on the rental unit",
    ],
}

# LTB file number patterns:
# Standard: TSL-12345-22, SOL-98765-23, TEL-00001-24-SA
# AI-generated: LTB-L-30001-25
CASE_NUMBER_RE = re.compile(
    r"\b([A-Z]{2,3}(?:-[A-Z])?-\d{4,6}-\d{2}(?:-[A-Z]{2,3})?)\b"
)

# Sentence splitting — split on period/newline but keep chunks meaningful
SENTENCE_RE = re.compile(r"(?<=[.!?\n])\s+")


def _load_model():
    """Load the sentence-transformers model. Downloads ~80MB on first run."""
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return model


def _split_into_cases(full_text):
    """Split extracted PDF text into individual case blocks by file number."""
    matches = list(CASE_NUMBER_RE.finditer(full_text))

    if not matches:
        # No file numbers found — treat as one block
        return [("FULL_DOC", full_text)]

    case_blocks = []
    seen = set()
    for i, m in enumerate(matches):
        case_id = m.group(1)
        if case_id in seen:
            continue
        seen.add(case_id)
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        block = full_text[start:end]
        if len(block.strip()) > 50:  # skip trivially small blocks
            case_blocks.append((case_id, block))

    return case_blocks


def _chunk_text(text, max_chunk_len=200):
    """Split case text into sentence-level chunks for embedding.

    We group sentences into chunks of roughly max_chunk_len characters
    to balance between granularity and speed.
    """
    sentences = SENTENCE_RE.split(text)
    chunks = []
    current = []
    current_len = 0

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        if current_len + len(sent) > max_chunk_len and current:
            chunks.append(" ".join(current))
            current = [sent]
            current_len = len(sent)
        else:
            current.append(sent)
            current_len += len(sent)

    if current:
        chunks.append(" ".join(current))

    return chunks


def analyze_cases(full_text, similarity_threshold=0.50, verbose=True):
    """Analyze extracted text and return list of cases with evidence types.

    Args:
        full_text: Full extracted text from PDF
        similarity_threshold: Min cosine similarity to count as a match (0-1).
            Lower = more matches, higher = stricter. 0.50 is well-tuned for
            LTB decisions.
        verbose: Print progress messages

    Returns:
        List of dicts: [{"case_id": "...", "evidence_types": [...]}, ...]
    """
    import numpy as np

    if not full_text or not full_text.strip():
        if verbose:
            print("No text to analyze.")
        return []

    if verbose:
        print("Loading semantic model (all-MiniLM-L6-v2)...")
    model = _load_model()

    # Pre-encode all reference phrases individually (not averaged).
    # Using max similarity across individual phrases is more precise than
    # comparing against an averaged category embedding.
    if verbose:
        print("Encoding evidence category references...")
    category_names = list(EVIDENCE_CATEGORIES.keys())

    all_ref_phrases = []
    phrase_to_cat_idx = []
    for cat_idx, cat_name in enumerate(category_names):
        for phrase in EVIDENCE_CATEGORIES[cat_name]:
            all_ref_phrases.append(phrase)
            phrase_to_cat_idx.append(cat_idx)

    ref_embeddings = model.encode(
        all_ref_phrases, show_progress_bar=False, normalize_embeddings=True
    )
    phrase_to_cat_idx = np.array(phrase_to_cat_idx)
    num_categories = len(category_names)

    # Split into cases
    case_blocks = _split_into_cases(full_text)
    if verbose:
        print(f"Found {len(case_blocks)} case(s). Analyzing...")

    all_cases = []
    for idx, (case_id, block) in enumerate(case_blocks):
        chunks = _chunk_text(block)
        if not chunks:
            all_cases.append({"case_id": case_id, "evidence_types": []})
            continue

        # Batch encode all chunks for this case
        chunk_embs = model.encode(chunks, show_progress_bar=False,
                                  normalize_embeddings=True, batch_size=64)

        # Cosine similarity: (num_chunks x dim) @ (dim x num_phrases)
        similarities = chunk_embs @ ref_embeddings.T  # (num_chunks, num_phrases)

        # For each category, find the max similarity across ALL its phrases
        # and ALL chunks. This is more precise than averaged embeddings.
        evidence_found = []
        for cat_i in range(num_categories):
            phrase_mask = phrase_to_cat_idx == cat_i
            cat_sims = similarities[:, phrase_mask]  # (num_chunks, phrases_in_cat)
            max_sim = cat_sims.max()
            if max_sim >= similarity_threshold:
                evidence_found.append(category_names[cat_i])

        all_cases.append({"case_id": case_id, "evidence_types": evidence_found})

        if verbose and (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(case_blocks)} cases...")

    if verbose:
        print(f"Analysis complete. {len(all_cases)} case(s) processed.")
        for c in all_cases[:5]:
            print(f"  {c['case_id']}: {len(c['evidence_types'])} evidence types")
        if len(all_cases) > 5:
            print(f"  ... and {len(all_cases) - 5} more")

    return all_cases


def aggregate_stats(all_cases):
    """Convert case-level results into the evidence_stats.json format.

    Returns:
        Dict in the format expected by the app's /evidence-stats endpoint.
    """
    total = len(all_cases)
    counts = Counter()
    for case in all_cases:
        for et in set(case.get("evidence_types", [])):
            counts[et] += 1

    evidence_types = []
    for category, count in counts.most_common():
        pct = round(100 * count / total) if total > 0 else 0
        evidence_types.append({
            "category": category,
            "percentage": pct,
            "count": count,
        })

    return {
        "N4": {
            "description": "Non-payment of rent",
            "total_cases_analyzed": total,
            "evidence_types": evidence_types,
        }
    }


def analyze_pdf(pdf_path, output_path=None, similarity_threshold=0.50):
    """Full pipeline: PDF → text extraction → analysis → JSON.

    Args:
        pdf_path: Path to concatenated PDF of LTB decisions
        output_path: Where to save evidence_stats.json (optional)
        similarity_threshold: Cosine similarity threshold (default 0.50)

    Returns:
        Dict with evidence statistics
    """
    import pdfplumber

    print(f"Extracting text from {pdf_path}...")
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)

    full_text = "\n\n".join(pages)
    print(f"Extracted {len(pages)} pages, {len(full_text):,} characters.")

    all_cases = analyze_cases(full_text, similarity_threshold=similarity_threshold)
    stats = aggregate_stats(all_cases)

    if output_path:
        with open(output_path, "w") as f:
            json.dump(stats, f, indent=2)
        print(f"\nSaved to {output_path}")

    return stats


# ---------------------------------------------------------------------------
# CLI usage
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python evidence_analyzer.py <pdf_path> [output.json]")
        sys.exit(1)

    pdf_file = sys.argv[1]
    out_file = sys.argv[2] if len(sys.argv) > 2 else "evidence_stats.json"
    stats = analyze_pdf(pdf_file, out_file)

    print("\nResults:")
    for ev in stats["N4"]["evidence_types"]:
        print(f"  {ev['percentage']:3d}%  ({ev['count']})  {ev['category']}")


## Step 5 (Optional): Run Evidence Analysis

Analyzes the uploaded LTB decision text and saves results to `evidence_stats.json`.

This JSON file powers the **Evidence Checklist** shown in the web app (Step 1 of the tool), which shows users what types of documents are most commonly submitted in cases like theirs.

You only need to run this once — re-run only if you have new decisions to analyze.

In [ ]:
# ─── Run Semantic Analysis ───────────────
import os, sys

STATS_PATH = '/content/evidence_stats.json'

if os.path.exists(STATS_PATH):
    print('evidence_stats.json already exists. Skipping analysis.')
    import json
    with open(STATS_PATH) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    sys.path.insert(0, '/content')
    from evidence_analyzer import analyze_cases, aggregate_stats

    all_cases = analyze_cases(full_text, similarity_threshold=0.50)
    stats = aggregate_stats(all_cases)

    import json
    with open(STATS_PATH, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f'\nSaved to {STATS_PATH}')

    # Show results
    total = stats['N4']['total_cases_analyzed']
    print(f'\nTotal cases analyzed: {total}')
    for ev in stats['N4']['evidence_types']:
        print(f"  {ev['percentage']:3d}%  ({ev['count']:3d}/{total})  {ev['category']}")

    try:
        from google.colab import files as colab_files
        colab_files.download(STATS_PATH)
    except Exception:
        pass


## Step 6: Write the Web App Frontend

Writes the complete HTML/CSS/JavaScript interface to `/content/templates/index.html`.

This is the 4-step wizard that users interact with:
1. **Case Information** — enter LTB file number, parties, hearing date, and case type
2. **Upload Files** — drag-and-drop upload for photos, PDFs, Word docs, and more
3. **Arrange Tabs** — organize files into named evidence tabs with drag-and-drop
4. **Review & Generate** — review the brief and download the formatted PDF

New in this version: collapsible guidance boxes on every step, suggested tab name chips, and a **✂ Redact** tool to black out sensitive information in images before generating the PDF.

In [ ]:
%%writefile /content/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Housekeeper</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Source+Serif+4:ital,opsz,wght@0,8..60,300;0,8..60,400;0,8..60,600;1,8..60,400&family=DM+Sans:wght@300;400;500;600&display=swap');

  :root {
    --navy: #1a3a5c;
    --navy-light: #2a5080;
    --gold: #c8a951;
    --gold-light: #e8c96a;
    --bg: #f4f6f9;
    --surface: #ffffff;
    --border: #d8dfe8;
    --text: #1e2b3a;
    --text-light: #6b7e94;
    --danger: #c0392b;
    --success: #27ae60;
    --radius: 10px;
    --shadow: 0 2px 12px rgba(26,58,92,0.10);
  }

  * { box-sizing: border-box; margin: 0; padding: 0; }

  body {
    font-family: 'DM Sans', sans-serif;
    background: var(--bg);
    color: var(--text);
    min-height: 100vh;
  }

  /* HEADER */
  header {
    background: var(--navy);
    color: white;
    padding: 0;
    box-shadow: 0 2px 16px rgba(0,0,0,0.18);
  }
  .header-inner {
    max-width: 1100px;
    margin: 0 auto;
    padding: 20px 32px;
    display: flex;
    align-items: center;
    gap: 18px;
  }
  .header-icon {
    background: var(--gold);
    border-radius: 8px;
    padding: 8px 12px;
    font-size: 22px;
  }
  header h1 {
    font-family: 'Source Serif 4', serif;
    font-size: 1.55rem;
    font-weight: 600;
    letter-spacing: -0.02em;
  }
  header p {
    font-size: 0.82rem;
    opacity: 0.65;
    margin-top: 2px;
  }
  .badge {
    margin-left: auto;
    background: rgba(200,169,81,0.18);
    color: var(--gold-light);
    border: 1px solid rgba(200,169,81,0.35);
    border-radius: 20px;
    padding: 4px 14px;
    font-size: 0.77rem;
    font-weight: 500;
    letter-spacing: 0.04em;
    white-space: nowrap;
  }

  /* PROGRESS STEPS */
  .steps {
    background: white;
    border-bottom: 1px solid var(--border);
  }
  .steps-inner {
    max-width: 1100px;
    margin: 0 auto;
    padding: 0 32px;
    display: flex;
  }
  .step-btn {
    padding: 14px 20px;
    font-family: 'DM Sans', sans-serif;
    font-size: 0.86rem;
    font-weight: 500;
    color: var(--text-light);
    background: none;
    border: none;
    border-bottom: 3px solid transparent;
    cursor: pointer;
    transition: all 0.2s;
    display: flex;
    align-items: center;
    gap: 8px;
  }
  .step-btn .num {
    width: 22px; height: 22px;
    border-radius: 50%;
    background: var(--border);
    color: var(--text-light);
    font-size: 0.75rem;
    display: flex; align-items: center; justify-content: center;
    font-weight: 600;
    transition: all 0.2s;
  }
  .step-btn.active {
    color: var(--navy);
    border-bottom-color: var(--gold);
  }
  .step-btn.active .num {
    background: var(--navy);
    color: white;
  }
  .step-btn.done .num {
    background: var(--success);
    color: white;
  }

  /* MAIN LAYOUT */
  .main {
    max-width: 1100px;
    margin: 0 auto;
    padding: 32px;
  }

  /* PANELS */
  .panel { display: none; }
  .panel.active { display: block; }

  /* FORM STYLES */
  .card {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    padding: 28px 32px;
    margin-bottom: 24px;
  }
  .card-title {
    font-family: 'Source Serif 4', serif;
    font-size: 1.1rem;
    font-weight: 600;
    color: var(--navy);
    margin-bottom: 20px;
    padding-bottom: 12px;
    border-bottom: 1px solid var(--border);
  }
  .form-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
  }
  .form-group {
    display: flex;
    flex-direction: column;
    gap: 6px;
  }
  .form-group.full { grid-column: 1 / -1; }
  label {
    font-size: 0.8rem;
    font-weight: 600;
    color: var(--text-light);
    text-transform: uppercase;
    letter-spacing: 0.05em;
  }
  input[type="text"], input[type="date"] {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.95rem;
    padding: 10px 14px;
    border: 1.5px solid var(--border);
    border-radius: 7px;
    color: var(--text);
    background: #fafbfc;
    transition: border-color 0.2s, box-shadow 0.2s;
    outline: none;
  }
  input[type="text"]:focus, input[type="date"]:focus {
    border-color: var(--navy-light);
    box-shadow: 0 0 0 3px rgba(26,58,92,0.08);
    background: white;
  }

  /* UPLOAD ZONE */
  .upload-zone {
    border: 2px dashed var(--border);
    border-radius: var(--radius);
    padding: 40px 24px;
    text-align: center;
    cursor: pointer;
    transition: all 0.2s;
    background: #fafbfc;
  }
  .upload-zone:hover, .upload-zone.dragover {
    border-color: var(--navy-light);
    background: rgba(26,58,92,0.04);
  }
  .upload-zone .icon { font-size: 2.5rem; margin-bottom: 12px; }
  .upload-zone p { color: var(--text-light); font-size: 0.9rem; }
  .upload-zone strong { color: var(--navy); }
  #file-input { display: none; }

  /* IMAGE GRID */
  #image-pool {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(120px, 1fr));
    gap: 12px;
    margin-top: 20px;
  }
  .img-card {
    border: 2px solid var(--border);
    border-radius: 8px;
    overflow: hidden;
    cursor: grab;
    transition: transform 0.15s, box-shadow 0.15s, border-color 0.15s;
    position: relative;
    background: #f0f3f8;
  }
  .img-card:hover { transform: translateY(-2px); box-shadow: 0 6px 20px rgba(0,0,0,0.12); }
  .img-card.dragging { opacity: 0.4; }
  .img-card img {
    width: 100%;
    aspect-ratio: 1;
    object-fit: cover;
    display: block;
  }
  .img-card .img-label {
    font-size: 0.68rem;
    padding: 4px 6px;
    color: var(--text-light);
    text-align: center;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
  }
  .img-card .remove-img {
    position: absolute;
    top: 4px; right: 4px;
    background: rgba(0,0,0,0.55);
    color: white;
    border: none;
    border-radius: 50%;
    width: 20px; height: 20px;
    font-size: 12px;
    cursor: pointer;
    display: none;
    align-items: center; justify-content: center;
    line-height: 1;
  }
  .img-card:hover .remove-img { display: flex; }
  .img-card.in-tab { border-color: var(--gold); }
  .img-card .tab-badge {
    position: absolute;
    bottom: 26px; left: 4px;
    background: var(--navy);
    color: white;
    font-size: 0.6rem;
    padding: 1px 5px;
    border-radius: 3px;
    font-weight: 600;
    display: none;
  }
  .img-card.in-tab .tab-badge { display: block; }

  /* FILE ICON (non-image uploads) */
  .file-icon-card {
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
    aspect-ratio: 1;
    width: 100%;
    background: #e8edf5;
    font-size: 2rem;
    gap: 4px;
  }
  .file-icon-card .file-ext {
    font-size: 0.55rem;
    font-weight: 700;
    text-transform: uppercase;
    color: var(--navy);
    background: rgba(26,58,92,0.15);
    padding: 1px 5px;
    border-radius: 2px;
  }

  /* TABS PANEL */
  .tabs-layout {
    display: grid;
    grid-template-columns: 280px 1fr;
    gap: 20px;
    align-items: start;
  }
  .tabs-sidebar {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    overflow: hidden;
  }
  .tabs-sidebar-header {
    background: var(--navy);
    color: white;
    padding: 14px 18px;
    font-size: 0.85rem;
    font-weight: 600;
    letter-spacing: 0.03em;
    display: flex;
    align-items: center;
    justify-content: space-between;
  }
  #tab-list { padding: 8px; }
  .tab-item {
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 10px 12px;
    border-radius: 7px;
    cursor: pointer;
    transition: background 0.15s;
    border: 1.5px solid transparent;
    margin-bottom: 4px;
  }
  .tab-item:hover { background: var(--bg); }
  .tab-item.selected {
    background: rgba(26,58,92,0.07);
    border-color: var(--navy);
  }
  .tab-item .tab-num {
    width: 28px; height: 28px;
    border-radius: 6px;
    background: var(--navy);
    color: white;
    font-size: 0.82rem;
    font-weight: 700;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0;
  }
  .tab-item .tab-name {
    font-size: 0.88rem;
    flex: 1;
    font-weight: 500;
  }
  .tab-item .tab-count {
    font-size: 0.75rem;
    color: var(--text-light);
    background: var(--bg);
    padding: 2px 8px;
    border-radius: 10px;
  }
  .tab-item .delete-tab {
    background: none;
    border: none;
    color: var(--text-light);
    cursor: pointer;
    font-size: 14px;
    padding: 2px;
    border-radius: 4px;
    opacity: 0;
    transition: opacity 0.15s, color 0.15s;
  }
  .tab-item:hover .delete-tab { opacity: 1; }
  .tab-item .delete-tab:hover { color: var(--danger); }

  #add-tab-btn {
    width: calc(100% - 16px);
    margin: 4px 8px 8px;
    padding: 9px;
    background: rgba(26,58,92,0.06);
    border: 1.5px dashed var(--navy-light);
    border-radius: 7px;
    color: var(--navy);
    font-family: 'DM Sans', sans-serif;
    font-size: 0.85rem;
    font-weight: 500;
    cursor: pointer;
    transition: background 0.15s;
  }
  #add-tab-btn:hover { background: rgba(26,58,92,0.12); }

  /* TAB DETAIL */
  .tab-detail {
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    box-shadow: var(--shadow);
    min-height: 400px;
  }
  .tab-detail-header {
    padding: 18px 24px;
    border-bottom: 1px solid var(--border);
    display: flex;
    align-items: center;
    gap: 16px;
  }
  .tab-detail-header .tab-num-big {
    width: 40px; height: 40px;
    border-radius: 10px;
    background: var(--navy);
    color: white;
    font-size: 1.1rem;
    font-weight: 700;
    display: flex; align-items: center; justify-content: center;
    flex-shrink: 0;
  }
  .tab-detail-header input[type="text"] {
    flex: 1;
    font-size: 1rem;
    font-weight: 600;
    padding: 8px 12px;
  }
  .tab-detail-body { padding: 20px 24px; }

  /* Drop zone inside tab */
  .tab-drop-zone {
    min-height: 160px;
    border: 2px dashed var(--border);
    border-radius: 8px;
    display: flex;
    align-items: center;
    justify-content: center;
    flex-direction: column;
    gap: 8px;
    color: var(--text-light);
    font-size: 0.85rem;
    transition: border-color 0.2s, background 0.2s;
    margin-bottom: 16px;
    padding: 16px;
  }
  .tab-drop-zone.dragover {
    border-color: var(--gold);
    background: rgba(200,169,81,0.06);
  }
  .tab-drop-zone.has-images { min-height: auto; padding: 0; border: none; }

  #tab-images-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(100px, 1fr));
    gap: 10px;
  }

  /* BUTTONS */
  .btn {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.9rem;
    font-weight: 600;
    padding: 11px 24px;
    border: none;
    border-radius: 8px;
    cursor: pointer;
    transition: all 0.18s;
    display: inline-flex;
    align-items: center;
    gap: 8px;
  }
  .btn-primary {
    background: var(--navy);
    color: white;
  }
  .btn-primary:hover { background: var(--navy-light); transform: translateY(-1px); box-shadow: 0 4px 14px rgba(26,58,92,0.25); }
  .btn-gold {
    background: var(--gold);
    color: var(--navy);
  }
  .btn-gold:hover { background: var(--gold-light); transform: translateY(-1px); box-shadow: 0 4px 14px rgba(200,169,81,0.35); }
  .btn-outline {
    background: white;
    color: var(--navy);
    border: 1.5px solid var(--border);
  }
  .btn-outline:hover { border-color: var(--navy); background: var(--bg); }
  .btn:disabled { opacity: 0.5; cursor: not-allowed; transform: none !important; }

  .btn-row {
    display: flex;
    gap: 12px;
    align-items: center;
    margin-top: 28px;
  }

  /* REVIEW PANEL */
  .review-section { margin-bottom: 24px; }
  .review-label {
    font-size: 0.78rem;
    font-weight: 700;
    color: var(--text-light);
    text-transform: uppercase;
    letter-spacing: 0.06em;
    margin-bottom: 8px;
  }
  .review-value {
    font-size: 0.95rem;
    color: var(--text);
  }
  .review-tabs { display: flex; flex-direction: column; gap: 8px; }
  .review-tab-row {
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 10px 16px;
    background: var(--bg);
    border-radius: 7px;
    font-size: 0.88rem;
  }
  .review-tab-row .rnum {
    font-weight: 700;
    color: var(--navy);
    min-width: 20px;
  }
  .review-tab-row .rtitle { flex: 1; }
  .review-tab-row .rcount {
    color: var(--text-light);
    font-size: 0.8rem;
  }

  /* GENERATE PANEL */
  .generate-center {
    text-align: center;
    padding: 40px 20px;
  }
  .generate-center .icon { font-size: 3.5rem; margin-bottom: 16px; }
  .generate-center h2 {
    font-family: 'Source Serif 4', serif;
    font-size: 1.4rem;
    color: var(--navy);
    margin-bottom: 8px;
  }
  .generate-center p { color: var(--text-light); font-size: 0.9rem; margin-bottom: 28px; }
  .spinner {
    width: 48px; height: 48px;
    border: 4px solid var(--border);
    border-top-color: var(--navy);
    border-radius: 50%;
    animation: spin 0.8s linear infinite;
    margin: 20px auto;
    display: none;
  }
  @keyframes spin { to { transform: rotate(360deg); } }

  /* UPLOAD PROGRESS */
  .upload-progress {
    margin-top: 12px;
    display: none;
  }
  .progress-bar {
    height: 4px;
    background: var(--border);
    border-radius: 2px;
    overflow: hidden;
  }
  .progress-fill {
    height: 100%;
    background: var(--navy);
    width: 0%;
    transition: width 0.3s;
  }
  .upload-status { font-size: 0.8rem; color: var(--text-light); margin-top: 6px; }

  /* NOTIFICATION */
  .notif {
    position: fixed;
    bottom: 24px; right: 24px;
    background: var(--navy);
    color: white;
    padding: 12px 20px;
    border-radius: 8px;
    font-size: 0.88rem;
    box-shadow: 0 6px 24px rgba(0,0,0,0.2);
    z-index: 999;
    transform: translateY(80px);
    opacity: 0;
    transition: all 0.3s;
  }
  .notif.show { transform: translateY(0); opacity: 1; }
  .notif.error { background: var(--danger); }

  /* SECTION INFO BOX */
  .info-box {
    background: rgba(26,58,92,0.06);
    border-left: 3px solid var(--navy);
    border-radius: 0 7px 7px 0;
    padding: 12px 16px;
    font-size: 0.84rem;
    color: var(--navy);
    margin-bottom: 20px;
  }

  /* CASE TYPE DROPDOWN + EVIDENCE STATS */
  select {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.95rem;
    padding: 10px 14px;
    border: 1.5px solid var(--border);
    border-radius: 7px;
    color: var(--text);
    background: #fafbfc;
    transition: border-color 0.2s, box-shadow 0.2s;
    outline: none;
    cursor: pointer;
    width: 100%;
  }
  select:focus {
    border-color: var(--navy-light);
    box-shadow: 0 0 0 3px rgba(26,58,92,0.08);
    background: white;
  }
  .evidence-stats-panel {
    margin-top: 16px;
    background: rgba(26,58,92,0.03);
    border: 1px solid var(--border);
    border-radius: var(--radius);
    padding: 16px 20px;
    display: none;
  }
  .evidence-stats-panel.visible { display: block; }
  .evidence-stats-panel h4 {
    font-family: 'Source Serif 4', serif;
    font-size: 0.95rem;
    font-weight: 600;
    color: var(--navy);
    margin: 0 0 12px;
  }
  .evidence-stats-note {
    font-size: 0.75rem;
    color: var(--text-light);
    font-style: italic;
    margin-bottom: 12px;
  }
  .evidence-stat-row {
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 6px 0;
    border-bottom: 1px solid rgba(0,0,0,0.04);
  }
  .evidence-stat-row:last-child { border-bottom: none; }
  .evidence-stat-bar {
    width: 60px; height: 6px;
    background: var(--border);
    border-radius: 3px;
    overflow: hidden;
    flex-shrink: 0;
  }
  .evidence-stat-bar-fill {
    height: 100%;
    background: var(--navy);
    border-radius: 3px;
    transition: width 0.4s ease;
  }
  .evidence-stat-pct {
    font-size: 0.82rem; font-weight: 600;
    color: var(--navy);
    min-width: 36px; text-align: right; flex-shrink: 0;
  }
  .evidence-stat-label {
    font-size: 0.84rem;
    color: var(--text);
    flex: 1;
  }

  /* MASCOT — ALEX THE RACCOON */
  .mascot-container {
    display: flex;
    align-items: center;
    gap: 14px;
    margin-bottom: 20px;
    animation: mascot-appear 0.4s ease;
  }
  @keyframes mascot-appear {
    from { opacity: 0; transform: translateY(8px); }
    to   { opacity: 1; transform: translateY(0); }
  }
  .raccoon-svg {
    width: 72px;
    height: 72px;
    flex-shrink: 0;
  }
  /* Speech bubble */
  .speech-bubble {
    background: #f0f4ff;
    border: 1px solid var(--border);
    border-radius: 12px;
    padding: 12px 16px;
    position: relative;
    flex: 1;
    font-size: 0.88rem;
    color: var(--navy);
    line-height: 1.5;
  }
  .speech-bubble::before {
    content: '';
    position: absolute;
    left: -8px;
    top: 18px;
    width: 0;
    height: 0;
    border-top: 6px solid transparent;
    border-bottom: 6px solid transparent;
    border-right: 8px solid var(--border);
  }
  .speech-bubble::after {
    content: '';
    position: absolute;
    left: -6px;
    top: 19px;
    width: 0;
    height: 0;
    border-top: 5px solid transparent;
    border-bottom: 5px solid transparent;
    border-right: 7px solid #f0f4ff;
  }
  .speech-bubble strong { color: var(--navy); }

  /* COLLAPSIBLE INFO BOX */
  .info-collapse {
    border: 1px solid var(--border);
    border-left: 3px solid var(--navy);
    border-radius: 0 8px 8px 0;
    margin-bottom: 20px;
    overflow: hidden;
  }
  .info-collapse-header {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 10px 14px;
    cursor: pointer;
    background: rgba(26,58,92,0.04);
    font-size: 0.84rem;
    font-weight: 600;
    color: var(--navy);
    user-select: none;
    gap: 8px;
  }
  .info-collapse-header:hover { background: rgba(26,58,92,0.08); }
  .info-collapse-arrow {
    font-size: 0.72rem;
    transition: transform 0.2s;
    color: var(--text-light);
    margin-left: auto;
  }
  .info-collapse.open .info-collapse-arrow { transform: rotate(180deg); }
  .info-collapse-body {
    display: none;
    padding: 12px 16px;
    font-size: 0.84rem;
    color: var(--text);
    line-height: 1.6;
    background: #fafbfc;
    border-top: 1px solid var(--border);
  }
  .info-collapse.open .info-collapse-body { display: block; }
  .info-collapse-body ul { margin: 6px 0 0 16px; }
  .info-collapse-body li { margin-bottom: 4px; }
  .info-collapse-body strong { color: var(--navy); }
  .info-collapse-body .disclaimer {
    margin-top: 10px;
    font-size: 0.79rem;
    color: var(--text-light);
    font-style: italic;
  }

  /* SUGGESTION CHIPS */
  .suggestion-chips {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-top: 14px;
    padding-top: 14px;
    border-top: 1px solid var(--border);
  }
  .suggestion-chips-label {
    font-size: 0.75rem;
    font-weight: 600;
    color: var(--text-light);
    text-transform: uppercase;
    letter-spacing: 0.05em;
    width: 100%;
    margin-bottom: 2px;
  }
  .chip-btn {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.78rem;
    font-weight: 500;
    padding: 5px 12px;
    background: white;
    border: 1.5px solid var(--navy);
    border-radius: 20px;
    color: var(--navy);
    cursor: pointer;
    transition: all 0.15s;
  }
  .chip-btn:hover { background: var(--navy); color: white; }
  .chip-btn.chip-added {
    background: rgba(26,58,92,0.12);
    border-color: var(--navy);
    color: var(--navy);
  }
  .chip-btn.chip-added:hover { background: var(--navy); color: white; }

  /* SUGGESTED TABS PANEL (Step 3) */
  .suggested-tabs-panel {
    background: rgba(200,169,81,0.07);
    border: 1px solid rgba(200,169,81,0.4);
    border-radius: var(--radius);
    padding: 14px 18px;
    margin-bottom: 20px;
    display: none;
  }
  .suggested-tabs-panel.visible { display: block; }
  .suggested-tabs-panel-title {
    font-size: 0.8rem;
    font-weight: 600;
    color: #8a6f1a;
    text-transform: uppercase;
    letter-spacing: 0.05em;
    margin-bottom: 10px;
  }

  /* TIP TEXT */
  .step-tip {
    font-size: 0.8rem;
    color: var(--text-light);
    font-style: italic;
    margin-bottom: 16px;
    margin-top: -8px;
  }

  /* REDACTION MODAL */
  .redact-modal-overlay {
    position: fixed;
    inset: 0;
    background: rgba(0,0,0,0.75);
    z-index: 1000;
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 20px;
  }
  .redact-modal-box {
    background: white;
    border-radius: 12px;
    box-shadow: 0 20px 60px rgba(0,0,0,0.4);
    max-width: 900px;
    width: 100%;
    max-height: 90vh;
    display: flex;
    flex-direction: column;
    overflow: hidden;
  }
  .redact-modal-header {
    padding: 14px 20px;
    background: var(--navy);
    color: white;
    display: flex;
    align-items: center;
    justify-content: space-between;
    gap: 12px;
    flex-shrink: 0;
  }
  .redact-modal-header span { font-weight: 600; font-size: 0.95rem; }
  .redact-modal-btns { display: flex; gap: 8px; }
  .redact-modal-btns button {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.82rem;
    font-weight: 500;
    padding: 7px 14px;
    border-radius: 6px;
    border: none;
    cursor: pointer;
    transition: opacity 0.15s;
  }
  .redact-modal-btns button:hover { opacity: 0.85; }
  .btn-undo { background: rgba(255,255,255,0.2); color: white; }
  .btn-save-redact { background: var(--gold); color: var(--navy); font-weight: 700; }
  .btn-cancel-redact { background: transparent; color: rgba(255,255,255,0.75); border: 1px solid rgba(255,255,255,0.35) !important; }
  .redact-hint {
    font-size: 0.82rem;
    color: var(--text-light);
    padding: 9px 20px;
    background: #fffbf0;
    border-bottom: 1px solid var(--border);
    flex-shrink: 0;
  }
  .redact-canvas-wrapper {
    overflow: auto;
    flex: 1;
    display: flex;
    align-items: center;
    justify-content: center;
    background: #2a2a2a;
    padding: 16px;
  }
  #redact-canvas { cursor: crosshair; display: block; max-width: 100%; }

  /* REDACT BUTTON on img-card */
  .redact-img {
    position: absolute;
    top: 4px; left: 4px;
    background: rgba(200,169,81,0.9);
    color: var(--navy);
    border: none;
    border-radius: 50%;
    width: 20px; height: 20px;
    font-size: 10px;
    cursor: pointer;
    display: none;
    align-items: center; justify-content: center;
    line-height: 1;
    font-weight: 700;
  }
  .img-card:hover .redact-img { display: flex; }

  /* REVIEW FILES GRID (Step 4) */
  .rev-files-grid {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(100px, 1fr));
    gap: 10px;
    margin-top: 8px;
    margin-bottom: 4px;
  }
  .rev-file-card {
    position: relative;
    border-radius: 8px;
    overflow: hidden;
    border: 1.5px solid var(--border);
    background: var(--surface);
  }
  .rev-file-card img {
    width: 100%;
    aspect-ratio: 1;
    object-fit: cover;
    display: block;
  }
  .rev-doc-icon {
    width: 100%;
    aspect-ratio: 1;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 2.2rem;
    background: #f4f4f8;
  }
  .rev-file-name {
    font-size: 0.7rem;
    padding: 4px 6px;
    color: var(--text);
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
    border-top: 1px solid var(--border);
  }
  .rev-redact-btn {
    position: absolute;
    top: 4px; right: 4px;
    background: rgba(200,169,81,0.92);
    color: var(--navy);
    border: none;
    border-radius: 5px;
    padding: 3px 8px;
    font-size: 0.75rem;
    font-weight: 700;
    cursor: pointer;
    opacity: 0;
    transition: opacity 0.15s;
    font-family: 'DM Sans', sans-serif;
  }
  .rev-file-card:hover .rev-redact-btn { opacity: 1; }

  /* PAGE NAVIGATION inside redact modal */
  .redact-page-nav {
    display: flex;
    align-items: center;
    gap: 12px;
    padding: 8px 20px;
    background: #1a2744;
    border-top: 1px solid rgba(255,255,255,0.12);
    flex-shrink: 0;
  }
  .redact-page-nav button {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.82rem;
    padding: 5px 13px;
    border-radius: 5px;
    border: none;
    background: rgba(255,255,255,0.18);
    color: white;
    cursor: pointer;
    transition: opacity 0.15s;
  }
  .redact-page-nav button:hover { opacity: 0.8; }
  .redact-page-nav button:disabled { opacity: 0.3; cursor: default; }
  .redact-page-nav span { color: rgba(255,255,255,0.85); font-size: 0.85rem; flex: 1; text-align: center; }

  /* WARNING MODAL */
  .warn-modal-overlay {
    position: fixed;
    inset: 0;
    background: rgba(0,0,0,0.55);
    z-index: 900;
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 20px;
  }
  .warn-modal-box {
    background: white;
    border-radius: 12px;
    box-shadow: 0 16px 48px rgba(0,0,0,0.3);
    max-width: 420px;
    width: 100%;
    padding: 32px 28px 24px;
    text-align: center;
  }
  .warn-icon { font-size: 2.4rem; margin-bottom: 12px; }
  .warn-modal-box h3 {
    font-family: 'Source Serif 4', serif;
    font-size: 1.1rem;
    color: var(--navy);
    margin-bottom: 10px;
  }
  .warn-modal-box p {
    font-size: 0.88rem;
    color: var(--text-light);
    line-height: 1.55;
    margin-bottom: 24px;
  }
  .warn-btns {
    display: flex;
    gap: 10px;
    justify-content: center;
  }
  .warn-btns button {
    font-family: 'DM Sans', sans-serif;
    font-size: 0.88rem;
    font-weight: 600;
    padding: 10px 20px;
    border-radius: 8px;
    border: none;
    cursor: pointer;
    transition: all 0.15s;
  }
  .warn-btn-back {
    background: var(--bg);
    color: var(--navy);
    border: 1.5px solid var(--border) !important;
  }
  .warn-btn-back:hover { border-color: var(--navy) !important; }
  .warn-btn-continue {
    background: var(--gold);
    color: var(--navy);
  }
  .warn-btn-continue:hover { background: var(--gold-light); }

  /* RESPONSIVE */
  @media (max-width: 720px) {
    .main { padding: 16px; }
    .form-grid { grid-template-columns: 1fr; }
    .tabs-layout { grid-template-columns: 1fr; }
  }
</style>
</head>
<body>

<header>
  <div class="header-inner">
    <div class="header-icon">⚖️</div>
    <div>
      <h1>Housekeeper</h1>
      <p>Landlord and Tenant Board — Hearing Preparation Tool</p>
    </div>
  </div>
</header>

<!-- Progress steps -->
<nav class="steps">
  <div class="steps-inner">
    <button class="step-btn active" onclick="goToStep(0)" id="step-btn-0">
      <span class="num">1</span> Case Information
    </button>
    <button class="step-btn" onclick="goToStep(1)" id="step-btn-1">
      <span class="num">2</span> Upload Files
    </button>
    <button class="step-btn" onclick="goToStep(2)" id="step-btn-2">
      <span class="num">3</span> Arrange Tabs
    </button>
    <button class="step-btn" onclick="goToStep(3)" id="step-btn-3">
      <span class="num">4</span> Review &amp; Generate
    </button>
  </div>
</nav>

<main class="main">

  <!-- STEP 1: CASE INFORMATION -->
  <div class="panel active" id="panel-0">
    <div class="card">
      <div class="card-title">Case Information</div>
      <div class="info-collapse open" id="what-is-brief-box">
        <div class="info-collapse-header" onclick="toggleInfoBox('what-is-brief-box')">
          <span>ℹ️ What is an Evidence Brief?</span>
          <span class="info-collapse-arrow">▼</span>
        </div>
        <div class="info-collapse-body">
          An <strong>Evidence Brief</strong> is a formal PDF document you submit to the Landlord and Tenant Board before your hearing. It organizes your supporting documents into numbered tabs, making it easy for the adjudicator to find evidence as you refer to it during the hearing.<br><br>
          Typically, each side submits their own brief. Label your tabs clearly — the adjudicator will follow along as you say things like <em>"I refer you to Tab 3, the lease agreement."</em>
        </div>
      </div>
      <div class="mascot-container" id="mascot-0">
        <svg class="raccoon-svg" viewBox="0 0 120 130" xmlns="http://www.w3.org/2000/svg">
          <!-- Tail (fluffy, striped, curling up behind) -->
          <path d="M88,100 Q108,95 115,78 Q120,62 112,50" stroke="#9e9e9e" stroke-width="10" fill="none" stroke-linecap="round"/>
          <path d="M93,95 Q110,88 116,74" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <path d="M105,78 Q114,65 113,55" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <!-- Body (round & chubby) -->
          <ellipse cx="58" cy="100" rx="30" ry="26" fill="#a8a8a8" stroke="#555" stroke-width="1.5"/>
          <!-- Belly (big light oval) -->
          <ellipse cx="58" cy="103" rx="20" ry="19" fill="#ddd"/>
          <!-- Feet (little round stumps) -->
          <ellipse cx="40" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="76" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <!-- Arms (little round paws at sides) -->
          <ellipse cx="30" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(-20,30,98)"/>
          <ellipse cx="86" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(20,86,98)"/>
          <!-- Ears (big round) -->
          <circle cx="28" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="88" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <!-- Ear inners (pink) -->
          <circle cx="28" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="88" cy="28" r="10" fill="#e8b4b8"/>
          <!-- Head (BIG round - chibi style) -->
          <circle cx="58" cy="50" r="34" fill="#b0b0b0" stroke="#555" stroke-width="1.5"/>
          <!-- White face area -->
          <ellipse cx="58" cy="56" rx="22" ry="18" fill="#e0e0e0"/>
          <!-- Eye mask patches (soft grey, raccoon signature) -->
          <ellipse cx="42" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <ellipse cx="74" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <!-- Eyes (BIG, cute, shiny) -->
          <circle cx="43" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="73" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <!-- Pupils (large, slightly offset up-right for cute look) -->
          <circle cx="45" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="75" cy="47" r="5" fill="#2a2a2a"/>
          <!-- Eye shine (two highlights per eye for sparkle) -->
          <circle cx="47" cy="44.5" r="2.2" fill="white"/>
          <circle cx="43" cy="49" r="1" fill="white"/>
          <circle cx="77" cy="44.5" r="2.2" fill="white"/>
          <circle cx="73" cy="49" r="1" fill="white"/>
          <!-- Nose (small round) -->
          <ellipse cx="58" cy="59" rx="3.5" ry="2.8" fill="#444"/>
          <ellipse cx="57" cy="58.5" rx="1.2" ry="0.7" fill="#777" opacity="0.5"/>
          <!-- Mouth (cute little w shape) -->
          <path d="M54,62 Q56,65 58,62 Q60,65 62,62" stroke="#666" stroke-width="1.2" fill="none" stroke-linecap="round"/>
          <!-- Cheek blush -->
          <circle cx="34" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <circle cx="82" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <!-- Forehead stripe (subtle) -->
          <path d="M50,22 L58,30 L66,22" stroke="#888" stroke-width="2" fill="none" stroke-linecap="round" opacity="0.4"/>
        </svg>
        <div class="speech-bubble" id="speech-0">
          <strong>Hi, I'm Alex the Mouse!</strong> Enter the party and file details exactly as they appear on your LTB documents.
        </div>
      </div>
      <div class="form-grid">
        <div class="form-group">
          <label>LTB File Number</label>
          <input type="text" id="file-number" placeholder="e.g. LTB-12345-24-OE">
        </div>
        <div class="form-group">
          <label>Hearing Date</label>
          <input type="date" id="hearing-date">
        </div>
        <div class="form-group">
          <label>Applicant Name</label>
          <input type="text" id="applicant-name" placeholder="Full legal name">
        </div>
        <div class="form-group">
          <label>Respondent Name</label>
          <input type="text" id="respondent-name" placeholder="Full legal name">
        </div>
        <div class="form-group full">
          <label>Applicant Address</label>
          <input type="text" id="applicant-address" placeholder="Street, City, Province, Postal Code">
        </div>
        <div class="form-group full">
          <label>Respondent / Rental Unit Address</label>
          <input type="text" id="respondent-address" placeholder="Street, City, Province, Postal Code">
        </div>
        <div class="form-group full" style="margin-top: 8px;">
          <label>Case Type (Informational)</label>
          <select id="case-type" onchange="onCaseTypeChange(this.value)">
            <option value="">— Select case type —</option>
            <option value="N4">N4 - Non-payment of rent</option>
            <option value="N5">N5 - Notice to end tenancy</option>
          </select>
        </div>
      </div>
      <div class="evidence-stats-panel" id="evidence-stats-panel">
        <h4>Evidence Checklist: What Adjudicators Commonly See in <span id="evidence-case-label">N4</span> Cases</h4>
        <div class="evidence-stats-note">Based on analysis of LTB decisions. Use this as a checklist when gathering your documents — and use these category names to label your tabs in Step 3.</div>
        <div id="evidence-stats-list"></div>
        <div class="suggestion-chips" id="suggestion-chips">
          <div class="suggestion-chips-label">Quick-add suggested tab names →</div>
        </div>
      </div>
    </div>
    <div class="btn-row">
      <button class="btn btn-primary" onclick="nextFromStep0()">Next: Upload Photos →</button>
    </div>
  </div>

  <!-- STEP 2: UPLOAD FILES -->
  <div class="panel" id="panel-1">
    <div class="card">
      <div class="card-title">Upload Files</div>
      <div class="mascot-container" id="mascot-1">
        <svg class="raccoon-svg" viewBox="0 0 120 130" xmlns="http://www.w3.org/2000/svg">
          <path d="M88,100 Q108,95 115,78 Q120,62 112,50" stroke="#9e9e9e" stroke-width="10" fill="none" stroke-linecap="round"/>
          <path d="M93,95 Q110,88 116,74" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <path d="M105,78 Q114,65 113,55" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <ellipse cx="58" cy="100" rx="30" ry="26" fill="#a8a8a8" stroke="#555" stroke-width="1.5"/>
          <ellipse cx="58" cy="103" rx="20" ry="19" fill="#ddd"/>
          <ellipse cx="40" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="76" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="30" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(-20,30,98)"/>
          <ellipse cx="86" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(20,86,98)"/>
          <circle cx="28" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="88" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="28" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="88" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="58" cy="50" r="34" fill="#b0b0b0" stroke="#555" stroke-width="1.5"/>
          <ellipse cx="58" cy="56" rx="22" ry="18" fill="#e0e0e0"/>
          <ellipse cx="42" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <ellipse cx="74" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <circle cx="43" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="73" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="45" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="75" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="47" cy="44.5" r="2.2" fill="white"/>
          <circle cx="43" cy="49" r="1" fill="white"/>
          <circle cx="77" cy="44.5" r="2.2" fill="white"/>
          <circle cx="73" cy="49" r="1" fill="white"/>
          <ellipse cx="58" cy="59" rx="3.5" ry="2.8" fill="#444"/>
          <ellipse cx="57" cy="58.5" rx="1.2" ry="0.7" fill="#777" opacity="0.5"/>
          <path d="M54,62 Q56,65 58,62 Q60,65 62,62" stroke="#666" stroke-width="1.2" fill="none" stroke-linecap="round"/>
          <circle cx="34" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <circle cx="82" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <path d="M50,22 L58,30 L66,22" stroke="#888" stroke-width="2" fill="none" stroke-linecap="round" opacity="0.4"/>
        </svg>
        <div class="speech-bubble">
          Upload all your supporting documents here — photos, your lease, rent receipts, email screenshots, inspection reports, and more. <strong>Higher quality photos make evidence clearer.</strong> I'll help you organize them into tabs in the next step.
        </div>
      </div>

      <div class="info-collapse open" id="upload-tips-box">
        <div class="info-collapse-header" onclick="toggleInfoBox('upload-tips-box')">
          <span>ℹ️ Tips for strong evidence</span>
          <span class="info-collapse-arrow">▼</span>
        </div>
        <div class="info-collapse-body">
          <ul>
            <li><strong>Photos:</strong> Clear, well-lit, and date-stamped if possible — show the specific issue</li>
            <li><strong>Documents:</strong> Include the <em>full</em> document, not just a highlighted page</li>
            <li><strong>Communications:</strong> Screenshot or export the full thread, not just one message</li>
            <li><strong>File formats:</strong> Images (JPG/PNG/HEIC), PDFs, Word docs, and most common formats are all supported</li>
            <li><strong>Sensitive info:</strong> Use the ✂ Redact button on any image to black out private details (e.g. SIN, banking info) before generating your PDF</li>
          </ul>
        </div>
      </div>

      <div class="upload-zone" id="upload-zone" onclick="document.getElementById('file-input').click()">
        <div class="icon">📁</div>
        <p><strong>Click to upload</strong> or drag &amp; drop files here</p>
        <p style="font-size:0.78rem; margin-top:4px;">All file types supported · Multiple files · Max 50 MB total</p>
      </div>
      <input type="file" id="file-input" multiple>

      <div class="upload-progress" id="upload-progress">
        <div class="progress-bar"><div class="progress-fill" id="progress-fill"></div></div>
        <div class="upload-status" id="upload-status">Uploading…</div>
      </div>

      <div id="image-pool"></div>
    </div>
    <div class="btn-row">
      <button class="btn btn-outline" onclick="goToStep(0)">← Back</button>
      <button class="btn btn-primary" onclick="proceedToTabs()">Next: Arrange Tabs →</button>

    </div>
  </div>

  <!-- STEP 3: ARRANGE TABS -->
  <div class="panel" id="panel-2">
    <div class="card">
      <div class="card-title">Arrange Tabs</div>
      <div class="mascot-container" id="mascot-2">
        <svg class="raccoon-svg" viewBox="0 0 120 130" xmlns="http://www.w3.org/2000/svg">
          <path d="M88,100 Q108,95 115,78 Q120,62 112,50" stroke="#9e9e9e" stroke-width="10" fill="none" stroke-linecap="round"/>
          <path d="M93,95 Q110,88 116,74" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <path d="M105,78 Q114,65 113,55" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <ellipse cx="58" cy="100" rx="30" ry="26" fill="#a8a8a8" stroke="#555" stroke-width="1.5"/>
          <ellipse cx="58" cy="103" rx="20" ry="19" fill="#ddd"/>
          <ellipse cx="40" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="76" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="30" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(-20,30,98)"/>
          <ellipse cx="86" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(20,86,98)"/>
          <circle cx="28" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="88" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="28" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="88" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="58" cy="50" r="34" fill="#b0b0b0" stroke="#555" stroke-width="1.5"/>
          <ellipse cx="58" cy="56" rx="22" ry="18" fill="#e0e0e0"/>
          <ellipse cx="42" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <ellipse cx="74" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <circle cx="43" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="73" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="45" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="75" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="47" cy="44.5" r="2.2" fill="white"/>
          <circle cx="43" cy="49" r="1" fill="white"/>
          <circle cx="77" cy="44.5" r="2.2" fill="white"/>
          <circle cx="73" cy="49" r="1" fill="white"/>
          <ellipse cx="58" cy="59" rx="3.5" ry="2.8" fill="#444"/>
          <ellipse cx="57" cy="58.5" rx="1.2" ry="0.7" fill="#777" opacity="0.5"/>
          <path d="M54,62 Q56,65 58,62 Q60,65 62,62" stroke="#666" stroke-width="1.2" fill="none" stroke-linecap="round"/>
          <circle cx="34" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <circle cx="82" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <path d="M50,22 L58,30 L66,22" stroke="#888" stroke-width="2" fill="none" stroke-linecap="round" opacity="0.4"/>
        </svg>
        <div class="speech-bubble">
          Each tab becomes a labeled section in your PDF. <strong>Adjudicators appreciate clearly named tabs</strong> — it makes it easy to find evidence when you reference it during the hearing.
        </div>
      </div>
      <p class="step-tip">Tip: Use descriptive names like "Lease Agreement" or "Rent Payment Records" rather than generic names like "Tab 1".</p>
      <div class="suggested-tabs-panel" id="suggested-tabs-panel">
        <div class="suggested-tabs-panel-title">Suggested tab names based on your case type — click to add:</div>
        <div class="suggestion-chips" id="step3-suggestion-chips" style="margin-top:0;padding-top:0;border-top:none;"></div>
      </div>
      <div class="tabs-layout">
        <!-- Sidebar: tab list -->
        <div class="tabs-sidebar">
          <div class="tabs-sidebar-header">
            <span>TABS</span>
            <span id="tab-count-badge" style="background:rgba(255,255,255,0.2);padding:2px 8px;border-radius:10px;font-size:0.78rem;">0</span>
          </div>
          <div id="tab-list"></div>
          <button id="add-tab-btn" onclick="addTab()">+ Add Tab</button>
        </div>

        <!-- Detail: selected tab -->
        <div class="tab-detail" id="tab-detail">
          <div style="padding:40px;text-align:center;color:var(--text-light);font-size:0.9rem;">
            ← Select a tab to add photos, or create your first tab
          </div>
        </div>
      </div>

      <!-- Pool of all uploaded images -->
      <div style="margin-top:24px;">
        <div class="card-title" style="margin-bottom:12px;">File Pool — drag files into a tab above</div>
        <div id="pool-grid" style="display:grid;grid-template-columns:repeat(auto-fill,minmax(100px,1fr));gap:10px;"></div>
        <p id="pool-empty" style="color:var(--text-light);font-size:0.85rem;display:none;">All files have been added to tabs.</p>
      </div>
    </div>
    <div class="btn-row">
      <button class="btn btn-outline" onclick="goToStep(1)">← Back</button>
      <button class="btn btn-primary" onclick="nextFromStep2()">Review &amp; Generate →</button>
    </div>
  </div>

  <!-- STEP 4: REVIEW & GENERATE -->
  <div class="panel" id="panel-3">
    <div class="card">
      <div class="card-title">Review Brief</div>
      <div class="mascot-container" id="mascot-3">
        <svg class="raccoon-svg" viewBox="0 0 120 130" xmlns="http://www.w3.org/2000/svg">
          <path d="M88,100 Q108,95 115,78 Q120,62 112,50" stroke="#9e9e9e" stroke-width="10" fill="none" stroke-linecap="round"/>
          <path d="M93,95 Q110,88 116,74" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <path d="M105,78 Q114,65 113,55" stroke="#7a7a7a" stroke-width="5" fill="none" stroke-linecap="round" opacity="0.5"/>
          <ellipse cx="58" cy="100" rx="30" ry="26" fill="#a8a8a8" stroke="#555" stroke-width="1.5"/>
          <ellipse cx="58" cy="103" rx="20" ry="19" fill="#ddd"/>
          <ellipse cx="40" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="76" cy="122" rx="10" ry="6" fill="#8a8a8a" stroke="#555" stroke-width="1.2"/>
          <ellipse cx="30" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(-20,30,98)"/>
          <ellipse cx="86" cy="98" rx="8" ry="6" fill="#9a9a9a" stroke="#555" stroke-width="1.2" transform="rotate(20,86,98)"/>
          <circle cx="28" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="88" cy="28" r="16" fill="#9e9e9e" stroke="#555" stroke-width="1.5"/>
          <circle cx="28" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="88" cy="28" r="10" fill="#e8b4b8"/>
          <circle cx="58" cy="50" r="34" fill="#b0b0b0" stroke="#555" stroke-width="1.5"/>
          <ellipse cx="58" cy="56" rx="22" ry="18" fill="#e0e0e0"/>
          <ellipse cx="42" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <ellipse cx="74" cy="48" rx="12" ry="9" fill="#777" opacity="0.6"/>
          <circle cx="43" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="73" cy="48" r="8" fill="white" stroke="#555" stroke-width="1"/>
          <circle cx="45" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="75" cy="47" r="5" fill="#2a2a2a"/>
          <circle cx="47" cy="44.5" r="2.2" fill="white"/>
          <circle cx="43" cy="49" r="1" fill="white"/>
          <circle cx="77" cy="44.5" r="2.2" fill="white"/>
          <circle cx="73" cy="49" r="1" fill="white"/>
          <ellipse cx="58" cy="59" rx="3.5" ry="2.8" fill="#444"/>
          <ellipse cx="57" cy="58.5" rx="1.2" ry="0.7" fill="#777" opacity="0.5"/>
          <path d="M54,62 Q56,65 58,62 Q60,65 62,62" stroke="#666" stroke-width="1.2" fill="none" stroke-linecap="round"/>
          <circle cx="34" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <circle cx="82" cy="58" r="5" fill="#e8a0a0" opacity="0.3"/>
          <path d="M50,22 L58,30 L66,22" stroke="#888" stroke-width="2" fill="none" stroke-linecap="round" opacity="0.4"/>
        </svg>
        <div class="speech-bubble">
          Looking good! Before generating, make sure all your tabs are <strong>correctly named and ordered</strong> — that's exactly how they'll appear in the PDF.
        </div>
      </div>

      <div class="info-collapse open" id="hearing-tips-box">
        <div class="info-collapse-header" onclick="toggleInfoBox('hearing-tips-box')">
          <span>📋 Attending your LTB Hearing — what to expect</span>
          <span class="info-collapse-arrow">▼</span>
        </div>
        <div class="info-collapse-body">
          <ul>
            <li><strong>Submit your Evidence Brief</strong> through the <strong>Ontario Tribunals Portal</strong> before your hearing deadline — do not bring printed copies</li>
            <li><strong>Serve the opposing party</strong> — deliver a copy of your Evidence Brief to the other party before the hearing (check your Notice of Hearing for the required deadline)</li>
            <li><strong>Join your hearing via Zoom</strong> — LTB hearings are conducted online; check your Notice of Hearing for the Zoom link and scheduled time</li>
            <li>Join <strong>10–15 minutes early</strong> to test your audio and video before the hearing starts</li>
            <li>Refer to your evidence by tab number during the hearing: <em>"I refer you to Tab 3, the lease agreement"</em></li>
            <li>Listen carefully and wait for the adjudicator to finish before responding</li>
          </ul>
          <p class="disclaimer">This tool helps you organize documents for your hearing. It is not legal advice. If available, consider speaking with a <strong>tenant duty counsel</strong> before your hearing.</p>
        </div>
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:24px;">
        <div>
          <div class="review-label">File Number</div>
          <div class="review-value" id="rev-file">—</div>
        </div>
        <div>
          <div class="review-label">Hearing Date</div>
          <div class="review-value" id="rev-date">—</div>
        </div>
        <div>
          <div class="review-label">Applicant</div>
          <div class="review-value" id="rev-applicant">—</div>
        </div>
        <div>
          <div class="review-label">Respondent</div>
          <div class="review-value" id="rev-respondent">—</div>
        </div>
        <div>
          <div class="review-label">Applicant Address</div>
          <div class="review-value" id="rev-app-addr">—</div>
        </div>
        <div>
          <div class="review-label">Respondent Address</div>
          <div class="review-value" id="rev-res-addr">—</div>
        </div>
      </div>

      <div class="review-label">Tabs</div>
      <div class="review-tabs" id="rev-tabs"></div>

      <div class="review-label" style="margin-top:20px;">Files <span style="font-weight:400;color:var(--text-light);font-size:0.82rem;">— hover a file and click ✂ to redact sensitive information before generating</span></div>
      <div id="rev-files-grid" class="rev-files-grid"></div>
    </div>

    <div class="card">
      <div class="generate-center">
        <div class="icon">📄</div>
        <h2>Ready to generate your Evidence Brief</h2>
        <p>Your PDF will include a title page, table of contents, and all photo tabs with page numbers.</p>
        <button class="btn btn-gold" id="generate-btn" onclick="generatePDF()" style="font-size:1rem;padding:14px 36px;">
          ⚡ Generate PDF
        </button>
        <div class="spinner" id="spinner"></div>
        <p id="gen-status" style="margin-top:12px;display:none;"></p>
      </div>
    </div>

    <div class="btn-row">
      <button class="btn btn-outline" onclick="goToStep(2)">← Back to Tabs</button>
    </div>
  </div>

</main>

<!-- Warning Modal -->
<div id="warn-modal" class="warn-modal-overlay" style="display:none">
  <div class="warn-modal-box">
    <div class="warn-icon">⚠️</div>
    <h3 id="warn-title">Some information is missing</h3>
    <p id="warn-message"></p>
    <div class="warn-btns">
      <button class="warn-btn-back" onclick="warnGoBack()">← Go Back</button>
      <button class="warn-btn-continue" onclick="warnContinue()">Skip &amp; Continue →</button>
    </div>
  </div>
</div>

<!-- Redaction Modal -->
<div id="redact-modal" class="redact-modal-overlay" style="display:none" onclick="handleRedactOverlayClick(event)">
  <div class="redact-modal-box" onclick="event.stopPropagation()">
    <div class="redact-modal-header">
      <span>✂ Redact Sensitive Information</span>
      <div class="redact-modal-btns">
        <button class="btn-undo" onclick="undoLastRedaction()">↩ Undo Last</button>
        <button class="btn-save-redact" onclick="saveRedaction()">Save Redaction</button>
        <button class="btn-cancel-redact" onclick="closeRedactModal()">Cancel</button>
      </div>
    </div>
    <div class="redact-hint">Click and drag on the image to draw black redaction boxes over any sensitive information you want to hide.</div>
    <div class="redact-canvas-wrapper">
      <canvas id="redact-canvas"></canvas>
    </div>
    <div class="redact-page-nav" id="redact-page-nav" style="display:none">
      <button id="redact-prev-btn" onclick="redactPrevPage()">‹ Prev</button>
      <span id="redact-page-label">Page 1 of 1</span>
      <button id="redact-next-btn" onclick="redactNextPage()">Next ›</button>
    </div>
  </div>
</div>

<!-- Notification -->
<div class="notif" id="notif"></div>

<script>
// ─── STATE ────────────────────────────────────────────────────────────────────
let uploadedImages = [];  // [{id, thumb, original}]
let tabs = [];            // [{id, title, imageIds:[]}]
let selectedTabId = null;
let previewPagesCache = {};   // fileId → {pages:[...], thumbs:[...]}
let redactPageBoxes = {};     // pageId → [{x,y,w,h},...] per-page box state
let currentRedactPages = [];  // page IDs for current doc (or [fileId] for images)
let currentRedactPageIndex = 0;
let currentStep = 0;

// ─── EVIDENCE STATS + GUIDANCE ───────────────────────────────────────────────
let evidenceStatsCache = null;
let suggestedTabNames = [];  // populated from case type, used in Step 3

const ALEX_DEFAULT_MSG = '<strong>Hi, I\'m Alex the Mouse!</strong> Enter the party and file details exactly as they appear on your LTB documents.';

// Collapsible info boxes
function toggleInfoBox(id) {
  document.getElementById(id).classList.toggle('open');
}

async function onCaseTypeChange(caseType) {
  const panel = document.getElementById('evidence-stats-panel');
  const speech = document.getElementById('speech-0');
  suggestedTabNames = [];

  if (!caseType) {
    panel.classList.remove('visible');
    speech.innerHTML = ALEX_DEFAULT_MSG;
    updateStep3SuggestionPanel();
    return;
  }

  // N5 — coming soon
  if (caseType === 'N5') {
    panel.classList.remove('visible');
    speech.innerHTML = '<strong>Heads up!</strong> We\'re still working on N5 stats — stay tuned! For now, you can still build your evidence brief as usual.';
    updateStep3SuggestionPanel();
    return;
  }

  if (!evidenceStatsCache) {
    try {
      const res = await fetch('/evidence-stats');
      evidenceStatsCache = await res.json();
    } catch (e) {
      console.error('Failed to load evidence stats:', e);
      panel.classList.remove('visible');
      return;
    }
  }

  const stats = evidenceStatsCache[caseType];
  if (!stats) { panel.classList.remove('visible'); return; }

  speech.innerHTML = '<strong>Great choice!</strong> Use the checklist below as a guide when gathering your documents — then name your tabs accordingly in Step 3.';

  document.getElementById('evidence-case-label').textContent =
    caseType + ' (' + stats.description + ')';

  document.getElementById('evidence-stats-list').innerHTML =
    stats.evidence_types.map(ev => `
      <div class="evidence-stat-row">
        <div class="evidence-stat-pct">${ev.percentage}%</div>
        <div class="evidence-stat-bar">
          <div class="evidence-stat-bar-fill" style="width:${ev.percentage}%"></div>
        </div>
        <div class="evidence-stat-label">${ev.category}</div>
      </div>`).join('');

  // Build suggestion chips from evidence category names
  suggestedTabNames = stats.evidence_types.map(ev => ev.category);
  renderSuggestionChips();
  updateStep3SuggestionPanel();

  panel.classList.add('visible');
}

function renderSuggestionChips() {
  const container = document.getElementById('suggestion-chips');
  container.innerHTML = '';
  const lbl = document.createElement('div');
  lbl.className = 'suggestion-chips-label';
  lbl.textContent = 'Quick-add suggested tab names →';
  container.appendChild(lbl);
  suggestedTabNames.forEach(name => {
    const alreadyHas = tabs.some(t => t.title === name);
    const btn = document.createElement('button');
    btn.className = 'chip-btn' + (alreadyHas ? ' chip-added' : '');
    btn.textContent = (alreadyHas ? '✓ ' : '+ ') + name;
    btn.title = alreadyHas ? 'Tab already added — click to go to Step 3' : 'Add as a tab name in Step 3';
    btn.onclick = () => addTabFromSuggestion(name);
    container.appendChild(btn);
  });
}

function addTabFromSuggestion(name) {
  const alreadyHas = tabs.some(t => t.title === name);
  if (!alreadyHas) {
    tabs.push({ id: 'tab_' + Date.now(), title: name, imageIds: [] });
  }
  // Refresh all chip sets
  renderSuggestionChips();
  updateStep3SuggestionPanel();
  notify(alreadyHas ? 'Tab "' + name + '" already exists' : '✓ Tab "' + name + '" added');
}

function updateStep3SuggestionPanel() {
  const panel = document.getElementById('suggested-tabs-panel');
  const container = document.getElementById('step3-suggestion-chips');
  if (!suggestedTabNames.length) {
    panel.classList.remove('visible');
    return;
  }
  panel.classList.add('visible');
  container.innerHTML = '';
  suggestedTabNames.forEach(name => {
    const alreadyHas = tabs.some(t => t.title === name);
    const btn = document.createElement('button');
    btn.className = 'chip-btn' + (alreadyHas ? ' chip-added' : '');
    btn.dataset.name = name;
    btn.textContent = (alreadyHas ? '✓ ' : '+ ') + name;
    btn.title = alreadyHas ? 'Tab exists — click to focus it' : 'Add tab with this name';
    btn.onclick = () => addNamedTab(name);
    container.appendChild(btn);
  });
}

function addNamedTab(name) {
  const existing = tabs.find(t => t.title === name);
  if (existing) {
    selectedTabId = existing.id;
    renderTabPanel();
    notify('Showing tab "' + name + '"');
  } else {
    const tab = { id: 'tab_' + Date.now(), title: name, imageIds: [] };
    tabs.push(tab);
    selectedTabId = tab.id;
    renderTabPanel();
    notify('✓ Tab "' + name + '" added');
  }
  updateStep3SuggestionPanel();
  renderSuggestionChips();
}

// ─── WARNING MODAL ───────────────────────────────────────────────────────────
let _warnCallback = null;

function showWarning(title, message, onContinue) {
  document.getElementById('warn-title').textContent = title;
  document.getElementById('warn-message').textContent = message;
  _warnCallback = onContinue;
  document.getElementById('warn-modal').style.display = 'flex';
}
function warnGoBack() {
  document.getElementById('warn-modal').style.display = 'none';
  _warnCallback = null;
}
function warnContinue() {
  document.getElementById('warn-modal').style.display = 'none';
  if (_warnCallback) _warnCallback();
  _warnCallback = null;
}

// ─── VALIDATED NAVIGATION ────────────────────────────────────────────────────
function nextFromStep0() {
  const missing = [];
  if (!document.getElementById('file-number').value.trim())  missing.push('LTB File Number');
  if (!document.getElementById('hearing-date').value)        missing.push('Hearing Date');
  if (!document.getElementById('applicant-name').value.trim()) missing.push('Applicant Name');
  if (!document.getElementById('respondent-name').value.trim()) missing.push('Respondent Name');

  if (missing.length) {
    showWarning(
      'Some fields are empty',
      'The following fields are blank: ' + missing.join(', ') + '. They will appear as blanks in your PDF. Do you want to go back and fill them in, or continue anyway?',
      () => goToStep(1)
    );
  } else {
    goToStep(1);
  }
}

function nextFromStep2() {
  const emptyTabs = tabs.filter(t => t.imageIds.length === 0);
  if (tabs.length === 0) {
    showWarning(
      'No tabs created',
      'You haven\'t created any evidence tabs yet. Your PDF will have no content. Do you want to go back and add tabs, or continue anyway?',
      () => goToStep(3)
    );
    return;
  }
  if (emptyTabs.length) {
    const names = emptyTabs.map((t, i) => t.title || ('Tab ' + (tabs.indexOf(t) + 1))).join(', ');
    showWarning(
      'Some tabs have no files',
      'The following tabs have no files assigned: ' + names + '. They will appear empty in your PDF. Continue anyway?',
      () => goToStep(3)
    );
  } else {
    goToStep(3);
  }
}

// ─── NAVIGATION ──────────────────────────────────────────────────────────────
function goToStep(step) {
  document.querySelectorAll('.panel').forEach((p, i) => p.classList.toggle('active', i === step));
  document.querySelectorAll('.step-btn').forEach((b, i) => {
    b.classList.toggle('active', i === step);
    if (i < step) b.querySelector('.num').textContent = '✓', b.classList.add('done');
    else b.querySelector('.num').textContent = i + 1, b.classList.remove('done');
  });
  currentStep = step;
  if (step === 2) { renderTabPanel(); updateStep3SuggestionPanel(); }
  if (step === 3) renderReview();
}

// ─── NOTIFICATIONS ───────────────────────────────────────────────────────────
function notify(msg, isError = false) {
  const el = document.getElementById('notif');
  el.textContent = msg;
  el.className = 'notif show' + (isError ? ' error' : '');
  setTimeout(() => el.classList.remove('show'), 3000);
}

// ─── UPLOAD ──────────────────────────────────────────────────────────────────
const uploadZone = document.getElementById('upload-zone');
uploadZone.addEventListener('dragover', e => { e.preventDefault(); uploadZone.classList.add('dragover'); });
uploadZone.addEventListener('dragleave', () => uploadZone.classList.remove('dragover'));
uploadZone.addEventListener('drop', e => {
  e.preventDefault();
  uploadZone.classList.remove('dragover');
  handleFiles(e.dataTransfer.files);
});
document.getElementById('file-input').addEventListener('change', e => handleFiles(e.target.files));

async function handleFiles(files) {
  if (!files.length) return;
  const formData = new FormData();
  for (const f of files) formData.append('files', f);

  document.getElementById('upload-progress').style.display = 'block';
  document.getElementById('progress-fill').style.width = '30%';
  document.getElementById('upload-status').textContent = `Uploading ${files.length} file(s)…`;

  try {
    const res = await fetch('/upload', { method: 'POST', body: formData });
    document.getElementById('progress-fill').style.width = '100%';
    const data = await res.json();
    if (data.error) throw new Error(data.error);

    uploadedImages.push(...data.files);
    renderImagePool();
    notify(`✓ ${data.files.length} file(s) uploaded`);
  } catch (e) {
    notify('Upload failed: ' + e.message, true);
  } finally {
    setTimeout(() => {
      document.getElementById('upload-progress').style.display = 'none';
      document.getElementById('progress-fill').style.width = '0%';
    }, 800);
    document.getElementById('file-input').value = '';
  }
}

function renderImagePool() {
  const pool = document.getElementById('image-pool');
  pool.innerHTML = '';
  uploadedImages.forEach((img, idx) => {
    const card = document.createElement('div');
    card.className = 'img-card';
    card.dataset.id = img.id;
    const assignedTab = tabs.find(t => t.imageIds.includes(img.id));
    if (assignedTab) {
      card.classList.add('in-tab');
    }
    const thumbSrc = `/thumbnail/${img.thumb || img.original}${img._cacheBust ? '?t=' + img._cacheBust : ''}`;
    const mediaHtml0 = img.is_image
      ? `<img src="${thumbSrc}" alt="File ${idx+1}" loading="lazy">`
      : `<div class="file-icon-card"><span>📄</span><span class="file-ext">${(img.name||img.id).split('.').pop().toLowerCase()}</span></div>`;
    card.innerHTML = `
      ${mediaHtml0}
      <div class="tab-badge">${assignedTab ? 'Tab ' + (tabs.indexOf(assignedTab)+1) : ''}</div>
      ${img.is_image ? `<button class="redact-img" onclick="openRedactModal('${img.id}',event)" title="Redact sensitive info">✂</button>` : ''}
      <button class="remove-img" onclick="removeImage('${img.id}',event)" title="Remove">×</button>
      <div class="img-label">${img.name || ('File ' + (idx+1))}</div>
    `;
    pool.appendChild(card);
  });
}

function removeImage(id, e) {
  e.stopPropagation();
  uploadedImages = uploadedImages.filter(i => i.id !== id);
  tabs.forEach(t => { t.imageIds = t.imageIds.filter(i => i !== id); });
  renderImagePool();
  if (currentStep === 2) renderTabPanel();
}

// ─── PROCEED TO TABS ─────────────────────────────────────────────────────────
function proceedToTabs() {
  if (uploadedImages.length === 0) {
    showWarning(
      'No files uploaded',
      'You haven\'t uploaded any files yet. Your evidence tabs will be empty and your PDF will have no content. Do you want to go back and upload files, or continue anyway?',
      () => { if (tabs.length === 0) addTab(); goToStep(2); }
    );
    return;
  }
  if (tabs.length === 0) addTab();
  goToStep(2);
}

// ─── TABS PANEL ──────────────────────────────────────────────────────────────
function addTab() {
  const tab = { id: 'tab_' + Date.now(), title: '', imageIds: [] };
  tabs.push(tab);
  selectedTabId = tab.id;
  renderTabPanel();
}

function deleteTab(id, e) {
  e.stopPropagation();
  tabs = tabs.filter(t => t.id !== id);
  if (selectedTabId === id) selectedTabId = tabs.length ? tabs[tabs.length - 1].id : null;
  renderTabPanel();
}

function selectTab(id) {
  selectedTabId = id;
  renderTabPanel();
}

function renderTabPanel() {
  // Tab count badge
  document.getElementById('tab-count-badge').textContent = tabs.length;

  // Tab list sidebar
  const list = document.getElementById('tab-list');
  list.innerHTML = '';
  tabs.forEach((tab, idx) => {
    const item = document.createElement('div');
    item.className = 'tab-item' + (tab.id === selectedTabId ? ' selected' : '');
    item.onclick = () => selectTab(tab.id);
    item.innerHTML = `
      <div class="tab-num">${idx + 1}</div>
      <div class="tab-name">${tab.title || `Tab ${idx+1}`}</div>
      <div class="tab-count">${tab.imageIds.length} 📎</div>
      <button class="delete-tab" onclick="deleteTab('${tab.id}',event)" title="Delete tab">🗑</button>
    `;
    list.appendChild(item);
  });

  // Tab detail
  const detail = document.getElementById('tab-detail');
  if (!selectedTabId || !tabs.find(t => t.id === selectedTabId)) {
    detail.innerHTML = `<div style="padding:40px;text-align:center;color:var(--text-light);font-size:0.9rem;">← Select a tab or click "Add Tab"</div>`;
  } else {
    const tab = tabs.find(t => t.id === selectedTabId);
    const tabIdx = tabs.indexOf(tab) + 1;
    detail.innerHTML = `
      <div class="tab-detail-header">
        <div class="tab-num-big">${tabIdx}</div>
        <input type="text" placeholder="Tab title (optional, e.g. 'Mould in Bathroom')"
          value="${tab.title}"
          oninput="tab_title_update(this.value)"
          style="flex:1;">
      </div>
      <div class="tab-detail-body">
        <p style="font-size:0.82rem;color:var(--text-light);margin-bottom:12px;">
          Drag files from the pool below into this area, or click files to add/remove them.
        </p>
        <div class="tab-drop-zone ${tab.imageIds.length ? 'has-images' : ''}" id="tab-drop-zone"
          ondragover="dzOver(event)" ondragleave="dzLeave(event)" ondrop="dzDrop(event,'${tab.id}')">
          ${tab.imageIds.length === 0
            ? '<div style="font-size:1.8rem">📥</div><div>Drop files here</div>'
            : ''}
          <div id="tab-images-grid">
            ${tab.imageIds.map((imgId, i) => {
              const img = uploadedImages.find(u => u.id === imgId);
              if (!img) return '';
              const ts = img._cacheBust ? '?t=' + img._cacheBust : '';
              const mediaHtml = img.is_image
                ? `<img src="/thumbnail/${img.thumb || img.original}${ts}" loading="lazy">`
                : `<div class="file-icon-card"><span>📄</span><span class="file-ext">${(img.name||img.id).split('.').pop().toLowerCase()}</span></div>`;
              return `
                <div class="img-card" draggable="true"
                  ondragstart="imgDragStart(event,'${imgId}')"
                  title="Click to remove from this tab">
                  ${mediaHtml}
                  ${img.is_image ? `<button class="redact-img" onclick="openRedactModal('${imgId}',event)" title="Redact sensitive info">✂</button>` : ''}
                  <button class="remove-img" onclick="removeFromTab('${tab.id}','${imgId}',event)" title="Remove">×</button>
                  <div class="img-label">${img.name || ('File ' + (i+1))}</div>
                </div>`;
            }).join('')}
          </div>
        </div>
      </div>
    `;
  }

  // Pool grid
  renderPoolGrid();
}

function tab_title_update(val) {
  const tab = tabs.find(t => t.id === selectedTabId);
  if (tab) tab.title = val;
  // Update sidebar label
  const list = document.getElementById('tab-list');
  const items = list.querySelectorAll('.tab-item');
  const idx = tabs.indexOf(tab);
  if (items[idx]) {
    items[idx].querySelector('.tab-name').textContent = val || `Tab ${idx+1}`;
  }
}

function renderPoolGrid() {
  const grid = document.getElementById('pool-grid');
  const unassigned = uploadedImages.filter(img =>
    !tabs.some(t => t.imageIds.includes(img.id))
  );

  document.getElementById('pool-empty').style.display = unassigned.length === 0 && uploadedImages.length > 0 ? 'block' : 'none';

  grid.innerHTML = unassigned.map((img, idx) => {
    const ts2 = img._cacheBust ? '?t=' + img._cacheBust : '';
    const mediaHtml = img.is_image
      ? `<img src="/thumbnail/${img.thumb || img.original}${ts2}" loading="lazy">`
      : `<div class="file-icon-card"><span>📄</span><span class="file-ext">${(img.name||img.id).split('.').pop().toLowerCase()}</span></div>`;
    return `
      <div class="img-card" draggable="true"
        ondragstart="imgDragStart(event,'${img.id}')"
        onclick="addToSelectedTab('${img.id}')"
        title="Click to add to selected tab / Drag to a tab">
        ${mediaHtml}
        ${img.is_image ? `<button class="redact-img" onclick="openRedactModal('${img.id}',event)" title="Redact sensitive info">✂</button>` : ''}
        <div class="img-label">${img.name || ('File ' + (uploadedImages.indexOf(img)+1))}</div>
      </div>`;
  }).join('');
}

// ─── DRAG AND DROP ───────────────────────────────────────────────────────────
let draggingId = null;

function imgDragStart(e, imgId) {
  draggingId = imgId;
  e.dataTransfer.effectAllowed = 'move';
  e.dataTransfer.setData('text/plain', imgId);
}

function dzOver(e) {
  e.preventDefault();
  document.getElementById('tab-drop-zone').classList.add('dragover');
}
function dzLeave(e) {
  document.getElementById('tab-drop-zone').classList.remove('dragover');
}
function dzDrop(e, tabId) {
  e.preventDefault();
  document.getElementById('tab-drop-zone').classList.remove('dragover');
  const imgId = e.dataTransfer.getData('text/plain') || draggingId;
  if (!imgId) return;
  addImageToTab(tabId, imgId);
}

function addImageToTab(tabId, imgId) {
  const tab = tabs.find(t => t.id === tabId);
  if (!tab) return;
  // Remove from any other tab first
  tabs.forEach(t => { t.imageIds = t.imageIds.filter(i => i !== imgId); });
  if (!tab.imageIds.includes(imgId)) tab.imageIds.push(imgId);
  renderTabPanel();
}

function addToSelectedTab(imgId) {
  if (!selectedTabId) { notify('Select a tab first.', true); return; }
  addImageToTab(selectedTabId, imgId);
}

function removeFromTab(tabId, imgId, e) {
  e.stopPropagation();
  const tab = tabs.find(t => t.id === tabId);
  if (tab) tab.imageIds = tab.imageIds.filter(i => i !== imgId);
  renderTabPanel();
}

// ─── REVIEW ──────────────────────────────────────────────────────────────────
function renderReview() {
  document.getElementById('rev-file').textContent = document.getElementById('file-number').value || '—';
  const d = document.getElementById('hearing-date').value;
  document.getElementById('rev-date').textContent = d ? new Date(d + 'T00:00').toLocaleDateString('en-CA', {year:'numeric',month:'long',day:'numeric'}) : '—';
  document.getElementById('rev-applicant').textContent = document.getElementById('applicant-name').value || '—';
  document.getElementById('rev-respondent').textContent = document.getElementById('respondent-name').value || '—';
  document.getElementById('rev-app-addr').textContent = document.getElementById('applicant-address').value || '—';
  document.getElementById('rev-res-addr').textContent = document.getElementById('respondent-address').value || '—';

  const revTabs = document.getElementById('rev-tabs');
  if (tabs.length === 0) {
    revTabs.innerHTML = '<p style="color:var(--text-light);font-size:0.85rem;">No tabs created yet.</p>';
  } else {
    revTabs.innerHTML = tabs.map((t, i) => `
      <div class="review-tab-row">
        <div class="rnum">${i+1}</div>
        <div class="rtitle">${t.title || `Tab ${i+1}`}</div>
        <div class="rcount">${t.imageIds.length} file${t.imageIds.length !== 1 ? 's' : ''}</div>
      </div>
    `).join('');
  }

  // Populate review files grid — unique files across all tabs, in order
  const revGrid = document.getElementById('rev-files-grid');
  if (revGrid) {
    const seen = new Set();
    const allFiles = [];
    tabs.forEach(t => t.imageIds.forEach(id => {
      if (!seen.has(id)) { seen.add(id); allFiles.push(id); }
    }));
    if (allFiles.length === 0) {
      revGrid.innerHTML = '<p style="color:var(--text-light);font-size:0.85rem;">No files added to tabs yet.</p>';
    } else {
      revGrid.innerHTML = allFiles.map(id => {
        const imgEntry = uploadedImages.find(u => u.id === id);
        if (!imgEntry) return '';
        const ts = imgEntry._cacheBust ? '?t=' + imgEntry._cacheBust : '';
        const mediaHtml = imgEntry.is_image
          ? `<img src="/thumbnail/${imgEntry.thumb || id}${ts}" alt="">`
          : `<div class="rev-doc-icon">${_docIcon(id)}</div>`;
        return `<div class="rev-file-card">
          ${mediaHtml}
          <div class="rev-file-name">${imgEntry.name || id}</div>
          <button class="rev-redact-btn" onclick="openRedactModal('${id}',event)" title="Redact sensitive info">✂</button>
        </div>`;
      }).join('');
    }
  }
}

function _docIcon(id) {
  const ext = (id.split('.').pop() || '').toLowerCase();
  if (ext === 'pdf') return '📄';
  if (ext === 'docx' || ext === 'doc') return '📝';
  if (ext === 'xlsx' || ext === 'xls') return '📊';
  if (ext === 'pptx' || ext === 'ppt') return '📑';
  return '📎';
}

// ─── REDACTION ───────────────────────────────────────────────────────────────
let currentRedactFileId = null;
let redactImage = null;
let redactionBoxes = [];
let redactDrawing = false;
let redactStart = { x: 0, y: 0 };
let redactCurrent = { x: 0, y: 0 };

function _loadRedactPageImage(pageId) {
  return new Promise(resolve => {
    const canvas = document.getElementById('redact-canvas');
    const image = new Image();
    image.onload = () => {
      redactImage = image;
      canvas.width = image.naturalWidth;
      canvas.height = image.naturalHeight;
      redactDrawCanvas();
      resolve();
    };
    image.onerror = resolve; // continue even on error
    image.src = '/thumbnail/' + pageId;
  });
}

function _updateRedactPageNav() {
  const nav = document.getElementById('redact-page-nav');
  const label = document.getElementById('redact-page-label');
  const prev = document.getElementById('redact-prev-btn');
  const next = document.getElementById('redact-next-btn');
  const total = currentRedactPages.length;
  if (total <= 1) { nav.style.display = 'none'; return; }
  nav.style.display = 'flex';
  label.textContent = `Page ${currentRedactPageIndex + 1} of ${total}`;
  prev.disabled = currentRedactPageIndex === 0;
  next.disabled = currentRedactPageIndex === total - 1;
}

async function redactGoToPage(idx) {
  // Save boxes for current page before switching
  if (currentRedactPages.length > 0) {
    redactPageBoxes[currentRedactPages[currentRedactPageIndex]] = [...redactionBoxes];
  }
  currentRedactPageIndex = idx;
  const pageId = currentRedactPages[idx];
  redactionBoxes = [...(redactPageBoxes[pageId] || [])];
  await _loadRedactPageImage(pageId);
  _updateRedactPageNav();
}

function redactPrevPage() { if (currentRedactPageIndex > 0) redactGoToPage(currentRedactPageIndex - 1); }
function redactNextPage() { if (currentRedactPageIndex < currentRedactPages.length - 1) redactGoToPage(currentRedactPageIndex + 1); }

async function openRedactModal(fileId, event) {
  if (event) event.stopPropagation();
  const imgEntry = uploadedImages.find(u => u.id === fileId);
  if (!imgEntry) return;

  currentRedactFileId = fileId;
  redactPageBoxes = {};
  redactionBoxes = [];
  currentRedactPageIndex = 0;

  if (imgEntry.is_image) {
    currentRedactPages = [fileId];
    document.getElementById('redact-page-nav').style.display = 'none';
    await _loadRedactPageImage(imgEntry.original || imgEntry.thumb || fileId);
  } else {
    // Document: fetch or use cached page list
    let pageData = previewPagesCache[fileId];
    if (!pageData) {
      notify('Loading document pages…');
      try {
        const r = await fetch('/preview-pages/' + fileId);
        const json = await r.json();
        if (!r.ok || json.error) throw new Error(json.error || 'Server error');
        pageData = json;
        previewPagesCache[fileId] = pageData;
      } catch (e) {
        notify('Cannot load file for redaction: ' + e.message, true);
        return;
      }
    }
    currentRedactPages = pageData.pages;
    await _loadRedactPageImage(currentRedactPages[0]);
    _updateRedactPageNav();
  }

  document.getElementById('redact-modal').style.display = 'flex';
}

function redactDrawCanvas() {
  const canvas = document.getElementById('redact-canvas');
  const ctx = canvas.getContext('2d');
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  if (redactImage) ctx.drawImage(redactImage, 0, 0);
  ctx.fillStyle = '#000';
  redactionBoxes.forEach(b => ctx.fillRect(b.x, b.y, b.w, b.h));
  if (redactDrawing) {
    const x = Math.min(redactStart.x, redactCurrent.x);
    const y = Math.min(redactStart.y, redactCurrent.y);
    const w = Math.abs(redactCurrent.x - redactStart.x);
    const h = Math.abs(redactCurrent.y - redactStart.y);
    ctx.fillStyle = 'rgba(0,0,0,0.6)';
    ctx.fillRect(x, y, w, h);
  }
}

function getCanvasPos(canvas, e) {
  const rect = canvas.getBoundingClientRect();
  const scaleX = canvas.width / rect.width;
  const scaleY = canvas.height / rect.height;
  const clientX = e.touches ? e.touches[0].clientX : e.clientX;
  const clientY = e.touches ? e.touches[0].clientY : e.clientY;
  return {
    x: (clientX - rect.left) * scaleX,
    y: (clientY - rect.top) * scaleY,
  };
}

document.getElementById('redact-canvas').addEventListener('mousedown', e => {
  redactDrawing = true;
  redactStart = getCanvasPos(e.target, e);
  redactCurrent = { ...redactStart };
});
document.getElementById('redact-canvas').addEventListener('mousemove', e => {
  if (!redactDrawing) return;
  redactCurrent = getCanvasPos(e.target, e);
  redactDrawCanvas();
});
document.addEventListener('mouseup', e => {
  if (!redactDrawing) return;
  redactDrawing = false;
  const x = Math.min(redactStart.x, redactCurrent.x);
  const y = Math.min(redactStart.y, redactCurrent.y);
  const w = Math.abs(redactCurrent.x - redactStart.x);
  const h = Math.abs(redactCurrent.y - redactStart.y);
  if (w > 3 && h > 3) redactionBoxes.push({ x, y, w, h });
  redactDrawCanvas();
});

function undoLastRedaction() {
  redactionBoxes.pop();
  redactDrawCanvas();
}

function closeRedactModal() {
  document.getElementById('redact-modal').style.display = 'none';
  currentRedactFileId = null;
  redactImage = null;
  redactionBoxes = [];
  redactPageBoxes = {};
  currentRedactPages = [];
  currentRedactPageIndex = 0;
}

function handleRedactOverlayClick(e) {
  if (e.target === document.getElementById('redact-modal')) closeRedactModal();
}

function _canvasToBlob(canvas) {
  return new Promise(resolve => canvas.toBlob(resolve, 'image/jpeg', 0.92));
}

async function _postRedactedCanvas(canvas, fileId) {
  const blob = await _canvasToBlob(canvas);
  const fd = new FormData();
  fd.append('file_id', fileId);
  fd.append('image', blob, fileId);
  const res = await fetch('/redact', { method: 'POST', body: fd });
  if (!res.ok) throw new Error('Server error');
  return res.json();
}

async function saveRedaction() {
  if (!currentRedactFileId) return;
  const saveBtn = document.querySelector('.btn-save-redact');
  saveBtn.disabled = true;
  saveBtn.textContent = 'Saving…';

  // Persist current page's boxes before saving
  if (currentRedactPages.length > 0) {
    redactPageBoxes[currentRedactPages[currentRedactPageIndex]] = [...redactionBoxes];
  }

  const imgEntry = uploadedImages.find(u => u.id === currentRedactFileId);

  try {
    if (imgEntry && imgEntry.is_image) {
      // Single image: save current canvas directly
      await _postRedactedCanvas(document.getElementById('redact-canvas'), currentRedactFileId);
      if (imgEntry) imgEntry._cacheBust = Date.now();
    } else {
      // Document: save each modified page using an offscreen canvas
      const pages = currentRedactPages;
      let savedCount = 0;
      for (let i = 0; i < pages.length; i++) {
        const pageId = pages[i];
        const boxes = redactPageBoxes[pageId] || [];
        if (boxes.length === 0) continue;
        // Re-render this page with its boxes on an offscreen canvas
        await new Promise((resolve, reject) => {
          const offImg = new Image();
          offImg.onload = async () => {
            const oc = document.createElement('canvas');
            oc.width = offImg.naturalWidth;
            oc.height = offImg.naturalHeight;
            const ctx = oc.getContext('2d');
            ctx.drawImage(offImg, 0, 0);
            ctx.fillStyle = '#000';
            boxes.forEach(b => ctx.fillRect(b.x, b.y, b.w, b.h));
            try { await _postRedactedCanvas(oc, pageId); savedCount++; resolve(); }
            catch (e) { reject(e); }
          };
          offImg.onerror = () => resolve(); // skip pages that fail to load
          offImg.src = '/thumbnail/' + pageId;
        });
      }
      if (savedCount === 0) {
        notify('No redaction boxes drawn — nothing to save.', true);
        saveBtn.disabled = false;
        saveBtn.textContent = 'Save Redaction';
        return;
      }
    }

    closeRedactModal();
    renderImagePool();
    if (currentStep === 2) renderTabPanel();
    if (currentStep === 3) renderReview();
    notify('✓ Redaction saved');
  } catch (err) {
    notify('Redaction failed: ' + err.message, true);
  } finally {
    saveBtn.disabled = false;
    saveBtn.textContent = 'Save Redaction';
  }
}

// ─── GENERATE PDF ────────────────────────────────────────────────────────────
async function generatePDF() {
  if (tabs.length === 0) { notify('Add at least one tab first.', true); return; }

  const btn = document.getElementById('generate-btn');
  const spinner = document.getElementById('spinner');
  const status = document.getElementById('gen-status');

  btn.disabled = true;
  spinner.style.display = 'block';
  status.style.display = 'block';
  status.textContent = 'Building your Evidence Brief…';
  status.style.color = 'var(--text-light)';

  // Build preview_pages map: for any non-image file that has cached page images,
  // tell the server to use those (potentially redacted) pages instead of re-converting
  const previewPagesPayload = {};
  tabs.forEach(t => t.imageIds.forEach(id => {
    if (previewPagesCache[id]) {
      previewPagesPayload[id] = previewPagesCache[id].pages;
    }
  }));

  const payload = {
    case_info: {
      file_number: document.getElementById('file-number').value,
      hearing_date: document.getElementById('hearing-date').value,
      applicant_name: document.getElementById('applicant-name').value,
      respondent_name: document.getElementById('respondent-name').value,
      applicant_address: document.getElementById('applicant-address').value,
      respondent_address: document.getElementById('respondent-address').value,
    },
    tabs: tabs.map(t => ({
      title: t.title,
      images: t.imageIds,
    })),
    preview_pages: previewPagesPayload,
  };

  try {
    const res = await fetch('/generate', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify(payload)
    });

    if (!res.ok) {
      let msg = 'Server error';
      try {
        const err = await res.json();
        msg = err.error || msg;
      } catch (_) {
        msg = `Server error (HTTP ${res.status})`;
      }
      throw new Error(msg);
    }

    const blob = await res.blob();
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = `Evidence_Brief_${payload.case_info.file_number || 'LTB'}.pdf`;
    a.click();
    URL.revokeObjectURL(url);

    status.textContent = '✓ PDF downloaded successfully!';
    status.style.color = 'var(--success)';
    notify('✓ Evidence Brief generated!');
  } catch (e) {
    status.textContent = 'Error: ' + e.message;
    status.style.color = 'var(--danger)';
    notify('Generation failed: ' + e.message, true);
  } finally {
    btn.disabled = false;
    spinner.style.display = 'none';
  }
}
</script>
</body>
</html>


## Step 7: Write the Flask Backend

Writes the Python web server to `/content/app.py`.

Key capabilities:
- **File uploads** — accepts images (JPG/PNG/HEIC/GIF/BMP/TIFF/WEBP), PDFs, Word, Excel, PowerPoint, and text files
- **Image processing** — auto-rotates mobile photos using EXIF data, generates thumbnails
- **Document conversion** — converts PDFs to images via PyMuPDF; Office files via LibreOffice
- **PDF generation** — two-pass ReportLab rendering for accurate table of contents page numbers
- **Redaction** — `POST /redact` endpoint lets users black out image areas before PDF generation

The server runs on port **5050**.

In [ ]:
%%writefile /content/app.py
# -*- coding: utf-8 -*-
import os
import json
import uuid
import io
import time
import secrets
import logging
import subprocess
from xml.sax.saxutils import escape as xml_escape
from flask import Flask, request, jsonify, send_file, render_template
from werkzeug.utils import secure_filename
from PIL import Image, ImageOps
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak,
    Image as RLImage, Table, TableStyle, HRFlowable
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.pdfgen import canvas
from reportlab.platypus.flowables import Flowable

# Optional text-extraction dependencies (app works without them)
try:
    import pdfplumber
    HAS_PDFPLUMBER = True
except ImportError:
    HAS_PDFPLUMBER = False

try:
    from docx import Document as DocxDocument
    HAS_DOCX = True
except ImportError:
    HAS_DOCX = False

try:
    import openpyxl
    HAS_OPENPYXL = True
except ImportError:
    HAS_OPENPYXL = False

try:
    import fitz as pymupdf
    HAS_PYMUPDF = True
except ImportError:
    HAS_PYMUPDF = False

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    HAS_PILLOW_HEIF = True
except ImportError:
    HAS_PILLOW_HEIF = False

app = Flask(__name__)
app.secret_key = os.environ.get('SECRET_KEY', secrets.token_hex(32))
app.config['UPLOAD_FOLDER'] = os.path.join(os.path.dirname(__file__), 'uploads')
app.config['MAX_CONTENT_LENGTH'] = 50 * 1024 * 1024  # 50MB

IMAGE_EXTS = {'png', 'jpg', 'jpeg', 'gif', 'bmp', 'tiff', 'webp'}
PLAIN_TEXT_EXTS = {
    'txt', 'md', 'csv', 'log', 'py', 'js', 'ts', 'html', 'htm',
    'css', 'json', 'xml', 'yaml', 'yml', 'ini', 'cfg', 'conf',
    'sql', 'sh', 'bat', 'tex', 'rst', 'rtf',
}
OFFICE_EXTS = {'docx', 'doc', 'xlsx', 'xls', 'pptx', 'ppt',
               'odt', 'ods', 'odp'}
HEIC_EXTS = {'heic', 'heif'}

def is_image_file(path):
    ext = path.rsplit('.', 1)[-1].lower() if '.' in path else ''
    return ext in IMAGE_EXTS


def extract_text_from_file(path):
    """Return (text, success). Reads the full extractable text with no truncation."""
    ext = path.rsplit('.', 1)[-1].lower() if '.' in path else ''
    try:
        if ext in PLAIN_TEXT_EXTS:
            with open(path, 'r', encoding='utf-8', errors='replace') as f:
                raw = f.read()
            return raw, True

        elif ext == 'pdf':
            if not HAS_PDFPLUMBER:
                return None, False
            parts = []
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    parts.append(page.extract_text() or '')
            return '\n'.join(parts), True

        elif ext == 'docx':
            if not HAS_DOCX:
                return None, False
            doc = DocxDocument(path)
            raw = '\n'.join(p.text for p in doc.paragraphs if p.text)
            return raw, True

        elif ext in ('xlsx', 'xls'):
            if not HAS_OPENPYXL:
                return None, False
            wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
            parts = []
            for sheet_name in wb.sheetnames:
                ws = wb[sheet_name]
                parts.append("── Sheet: %s ──" % sheet_name)
                for row in ws.iter_rows(values_only=True):
                    cells_list = [str(c) if c is not None else '' for c in row]
                    parts.append('\t'.join(cells_list))
            wb.close()
            return '\n'.join(parts), True

        else:
            return None, False

    except Exception:
        return None, False


# ─── DOCUMENT-TO-PDF CONVERSION ───────────────────────────────────────────────

def _find_libreoffice():
    """Locate LibreOffice / soffice executable."""
    for cmd in ['soffice', 'libreoffice',
                '/Applications/LibreOffice.app/Contents/MacOS/soffice']:
        try:
            subprocess.run([cmd, '--version'], capture_output=True, timeout=10)
            return cmd
        except (FileNotFoundError, subprocess.TimeoutExpired):
            continue
    return None

_LO_CMD_CACHE = [None, False]   # [command, searched]

def _get_lo_cmd():
    if not _LO_CMD_CACHE[1]:
        _LO_CMD_CACHE[0] = _find_libreoffice()
        _LO_CMD_CACHE[1] = True
    return _LO_CMD_CACHE[0]


def _convert_heic_to_jpeg(src_path, dest_path):
    """Convert a HEIC/HEIF file to JPEG.
    Tries pillow-heif first; falls back to macOS sips."""
    if HAS_PILLOW_HEIF:
        try:
            with Image.open(src_path) as img:
                img.convert('RGB').save(dest_path, 'JPEG', quality=95)
            return os.path.exists(dest_path)
        except Exception:
            pass
    try:
        r = subprocess.run(
            ['sips', '-s', 'format', 'jpeg', src_path, '--out', dest_path],
            capture_output=True, timeout=60,
        )
        return r.returncode == 0 and os.path.exists(dest_path)
    except Exception:
        return False


def _convert_office_to_pdf(file_path):
    """Convert an office document to PDF via LibreOffice headless."""
    lo_cmd = _get_lo_cmd()
    if not lo_cmd:
        return None
    try:
        outdir = app.config['UPLOAD_FOLDER']
        # Use a unique user profile to avoid lock conflicts between workers
        user_profile = os.path.join(outdir, 'lo_profile_%s' % uuid.uuid4().hex)
        subprocess.run(
            [lo_cmd, '--headless',
             '-env:UserInstallation=file://' + user_profile,
             '--convert-to', 'pdf',
             '--outdir', outdir, file_path],
            capture_output=True, timeout=120,
        )
        # Clean up the temporary profile
        import shutil
        shutil.rmtree(user_profile, ignore_errors=True)
        base = os.path.splitext(os.path.basename(file_path))[0]
        out_path = os.path.join(outdir, base + '.pdf')
        if os.path.exists(out_path):
            new_path = os.path.join(outdir, 'conv_%s.pdf' % uuid.uuid4().hex)
            os.rename(out_path, new_path)
            return new_path
    except Exception:
        pass
    return None


def _convert_text_to_pdf(file_path):
    """Render a plain-text file as a PDF using ReportLab. Returns path or None."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            text = f.read()
    except Exception:
        return None
    if not text.strip():
        return None
    out_path = os.path.join(
        app.config['UPLOAD_FOLDER'],
        'textconv_%s.pdf' % uuid.uuid4().hex,
    )
    try:
        doc = SimpleDocTemplate(
            out_path, pagesize=letter,
            leftMargin=0.6 * inch, rightMargin=0.6 * inch,
            topMargin=0.6 * inch, bottomMargin=0.6 * inch,
        )
        styles = getSampleStyleSheet()
        code_style = ParagraphStyle(
            'TextConv', parent=styles['Normal'],
            fontName='Courier', fontSize=8, leading=10,
            spaceBefore=0, spaceAfter=0,
        )
        story = []
        for line in text.split('\n'):
            safe = xml_escape(line) if line.strip() else '&nbsp;'
            story.append(Paragraph(safe, code_style))
        doc.build(story)
        return out_path
    except Exception:
        try:
            os.unlink(out_path)
        except OSError:
            pass
        return None


def _convert_docx_to_pdf_reportlab(file_path):
    """Convert .docx to PDF using python-docx, preserving text and embedded images.
    Walks the document body XML in order so images appear in context with surrounding text."""
    if not HAS_DOCX:
        return None
    try:
        doc = DocxDocument(file_path)
    except Exception:
        return None

    # XML namespaces used in docx/drawingml
    _A = 'http://schemas.openxmlformats.org/drawingml/2006/main'
    _R = 'http://schemas.openxmlformats.org/officeDocument/2006/relationships'
    _W = 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'

    out_path = os.path.join(
        app.config['UPLOAD_FOLDER'],
        'docxconv_%s.pdf' % uuid.uuid4().hex,
    )

    rl_styles = getSampleStyleSheet()
    normal_s = ParagraphStyle('DxBody', parent=rl_styles['Normal'],
                              fontName='Helvetica', fontSize=10, leading=14, spaceAfter=3)
    h1_s = ParagraphStyle('DxH1', parent=rl_styles['Normal'],
                           fontName='Helvetica-Bold', fontSize=14, leading=18,
                           spaceBefore=10, spaceAfter=4)
    h2_s = ParagraphStyle('DxH2', parent=rl_styles['Normal'],
                           fontName='Helvetica-Bold', fontSize=12, leading=16,
                           spaceBefore=8, spaceAfter=3)

    usable_w = letter[0] - 1.5 * inch

    def _image_flowable(blob):
        try:
            buf = io.BytesIO(blob)
            with Image.open(buf) as pil:
                orig_w, orig_h = pil.size
                mode = pil.mode
            # ReportLab struggles with RGBA/palette modes — normalise to RGB JPEG
            if mode in ('RGBA', 'P', 'LA'):
                buf.seek(0)
                out_buf = io.BytesIO()
                with Image.open(buf) as pil:
                    pil.convert('RGB').save(out_buf, 'JPEG', quality=90)
                out_buf.seek(0)
                buf = out_buf
            else:
                buf.seek(0)
            scale = min(usable_w / orig_w, 4 * inch / orig_h, 1.0)
            rl = RLImage(buf, width=orig_w * scale, height=orig_h * scale)
            rl.hAlign = 'CENTER'
            return rl
        except Exception:
            return None

    def _para_flowables(p_elem):
        """Return flowables for a single <w:p> element (images + text)."""
        items = []
        for blip in p_elem.iter('{%s}blip' % _A):
            rId = blip.get('{%s}embed' % _R)
            if rId:
                try:
                    blob = doc.part.related_parts[rId].blob
                    fl = _image_flowable(blob)
                    if fl:
                        items.append(fl)
                        items.append(Spacer(1, 0.1 * inch))
                except Exception:
                    pass
        texts = [t.text or '' for t in p_elem.iter('{%s}t' % _W)]
        text = ''.join(texts).strip()
        if text:
            ps_elem = p_elem.find('.//{%s}pStyle' % _W)
            sv = ps_elem.get('{%s}val' % _W, '') if ps_elem is not None else ''
            if 'Heading1' in sv or 'Title' in sv:
                ps = h1_s
            elif 'Heading' in sv:
                ps = h2_s
            else:
                ps = normal_s
            items.append(Paragraph(xml_escape(text), ps))
        return items

    try:
        story = []
        for child in doc.element.body:
            local = child.tag.split('}')[-1] if '}' in child.tag else child.tag
            if local == 'p':
                story.extend(_para_flowables(child))
            elif local == 'tbl':
                for tr in child.iter('{%s}tr' % _W):
                    for tc in tr.findall('{%s}tc' % _W):
                        for p in tc.findall('{%s}p' % _W):
                            story.extend(_para_flowables(p))
        if not story:
            return None
        rl_doc = SimpleDocTemplate(
            out_path, pagesize=letter,
            leftMargin=0.75 * inch, rightMargin=0.75 * inch,
            topMargin=0.75 * inch, bottomMargin=0.75 * inch,
        )
        rl_doc.build(story)
        return out_path
    except Exception:
        try:
            os.unlink(out_path)
        except OSError:
            pass
        return None


def convert_document_to_images(file_path):
    """Convert ANY non-image file to rendered page images using PyMuPDF.

    Handles PDFs directly, office docs via LibreOffice, and text files
    via ReportLab. Returns a list of image file paths (one per page),
    or an empty list if conversion is not possible.
    """
    ext = file_path.rsplit('.', 1)[-1].lower() if '.' in file_path else ''
    if ext in IMAGE_EXTS or not HAS_PYMUPDF:
        return []

    pdf_path = file_path
    cleanup_pdf = False

    if ext != 'pdf':
        converted = None
        if ext in OFFICE_EXTS:
            converted = _convert_office_to_pdf(file_path)
        if not converted and ext in PLAIN_TEXT_EXTS:
            converted = _convert_text_to_pdf(file_path)
        if not converted and ext in ('docx', 'doc'):
            converted = _convert_docx_to_pdf_reportlab(file_path)
        if not converted and ext in OFFICE_EXTS:
            # Office conversion failed and it's not a text file — give up
            return []
        if not converted:
            return []
        pdf_path = converted
        cleanup_pdf = True

    try:
        doc = pymupdf.open(pdf_path)
        images = []
        for page in doc:
            pix = page.get_pixmap(dpi=150)
            img_path = os.path.join(
                app.config['UPLOAD_FOLDER'],
                'docpage_%s.jpg' % uuid.uuid4().hex,
            )
            mode = 'RGBA' if pix.alpha else 'RGB'
            pil_page = Image.frombytes(mode, (pix.width, pix.height), pix.samples)
            if pil_page.mode != 'RGB':
                pil_page = pil_page.convert('RGB')
            pil_page.save(img_path, 'JPEG', quality=72)
            images.append(img_path)
        doc.close()
        return images
    except Exception:
        return []
    finally:
        if cleanup_pdf:
            try:
                os.unlink(pdf_path)
            except OSError:
                pass


# ─── PAGE NUMBERING ────────────────────────────────────────────────────────────

class NumberedCanvas(canvas.Canvas):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._saved_page_states = []

    def showPage(self):
        self._saved_page_states.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        for state in self._saved_page_states:
            self.__dict__.update(state)
            self.draw_page_footer()
            super().showPage()
        super().save()

    def draw_page_footer(self):
        self.setFont("Times-Bold", 14)
        self.setFillColor(colors.HexColor("#333333"))
        width, height = letter
        self.drawRightString(width - 0.5 * inch, 0.35 * inch, "Page %d" % self._pageNumber)
        self.setFont("Times-Roman", 11)
        self.setFillColor(colors.HexColor("#666666"))
        self.drawString(0.5 * inch, 0.35 * inch, "Evidence Brief")
        self.setStrokeColor(colors.HexColor("#cccccc"))
        self.setLineWidth(0.5)
        self.line(0.5 * inch, 0.6 * inch, width - 0.5 * inch, 0.6 * inch)


# ─── TAB DIVIDER FLOWABLE ──────────────────────────────────────────────────────

class TabDivider(Flowable):
    """Blank tab-divider page.
    Records self.canv._pageNumber in page_registry on first draw (TOC two-pass).
    """
    def __init__(self, tab_number, tab_title, width, height,
                 page_registry=None, registry_key=None):
        super().__init__()
        self.tab_number = tab_number
        self.tab_title = tab_title
        self.width = width
        self.height = height
        self.page_registry = page_registry
        self.registry_key = registry_key if registry_key is not None else tab_number

    def draw(self):
        # Record page number for TOC (pass 1 only)
        if self.page_registry is not None:
            self.page_registry[self.registry_key] = self.canv._pageNumber

        cx = self.width / 2
        cy = self.height / 2

        # "TAB N" centered on page — no lines, no description
        self.canv.setFillColor(colors.black)
        self.canv.setFont("Times-Bold", 28)
        self.canv.drawCentredString(cx, cy, "TAB %d" % self.tab_number)

    def wrap(self, availWidth, availHeight):
        return self.width, self.height


# ─── IMAGE COMPRESSION HELPER ──────────────────────────────────────────────────

def _rl_image_compressed(img_path, max_w_pts, max_h_pts, quality=72):
    """Return a centred RLImage downscaled to 150 DPI at its display size and
    re-encoded as JPEG.  This keeps individual image contributions small without
    visible quality loss at typical screen/print sizes."""
    TARGET_DPI = 150
    PTS_PER_INCH = 72.0
    with Image.open(img_path) as pil:
        orig_w, orig_h = pil.size
        scale = min(max_w_pts / orig_w, max_h_pts / orig_h, 1.0)
        disp_w = orig_w * scale   # points
        disp_h = orig_h * scale
        # Pixel dimensions needed for TARGET_DPI at the display size
        px_w = max(1, int(disp_w / PTS_PER_INCH * TARGET_DPI))
        px_h = max(1, int(disp_h / PTS_PER_INCH * TARGET_DPI))
        img = pil.copy()
    # Downscale only — never upscale
    if img.width > px_w or img.height > px_h:
        img = img.resize((px_w, px_h), Image.LANCZOS)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    buf = io.BytesIO()
    img.save(buf, 'JPEG', quality=quality, optimize=True)
    buf.seek(0)
    rl = RLImage(buf, width=disp_w, height=disp_h)
    rl.hAlign = 'CENTER'
    return rl


# ─── STORY BUILDER ─────────────────────────────────────────────────────────────

def _build_story(case_info, tabs_data, usable_w, usable_h,
                 page_registry=None, tab_pages=None, text_cache=None,
                 doc_page_images=None):
    """
    Build and return the full ReportLab story.

    page_registry   – dict populated with {tab_idx: page_number} during draw (pass 1).
    tab_pages       – dict {tab_idx: page_number} used to fill the TOC (pass 2).
    text_cache      – shared dict {path: (text, truncated, ok)} to avoid double extraction.
    doc_page_images – dict {path: [img_path, …]} of pre-rendered document pages.
    """
    if text_cache is None:
        text_cache = {}

    styles = getSampleStyleSheet()
    story = []

    # ── TITLE PAGE (LTB template style) ──────────────────────────────────────
    # Paragraph styles — Times New Roman throughout, matching the LTB template
    tn_r  = ParagraphStyle('TnR',  parent=styles['Normal'],
                           fontName='Times-Roman',  fontSize=12, leading=18)
    tn_b  = ParagraphStyle('TnB',  parent=styles['Normal'],
                           fontName='Times-Bold',   fontSize=12, leading=18)
    tn_rc = ParagraphStyle('TnRc', parent=styles['Normal'],
                           fontName='Times-Roman',  fontSize=12, leading=18,
                           alignment=TA_CENTER)
    tn_bc = ParagraphStyle('TnBc', parent=styles['Normal'],
                           fontName='Times-Bold',   fontSize=12, leading=18,
                           alignment=TA_CENTER)
    tn_rr = ParagraphStyle('TnRr', parent=styles['Normal'],
                           fontName='Times-Roman',  fontSize=12, leading=18,
                           alignment=TA_RIGHT)

    # File number — top right
    fn = case_info.get('file_number', '')
    if fn:
        story.append(Paragraph("File Number: " + xml_escape(fn), tn_rr))
    story.append(Spacer(1, 0.45 * inch))

    # LANDLORD AND TENANT BOARD / TRIBUNALS ONTARIO
    story.append(Paragraph("LANDLORD AND TENANT BOARD", tn_bc))
    story.append(Paragraph("TRIBUNALS ONTARIO", tn_bc))
    story.append(Spacer(1, 0.35 * inch))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.black))
    story.append(Spacer(1, 0.3 * inch))

    # In the matter of (rental unit address = respondent_address)
    unit_addr = case_info.get('respondent_address', '')
    if unit_addr:
        story.append(Paragraph("In the matter of: " + xml_escape(unit_addr), tn_r))
        story.append(Spacer(1, 0.3 * inch))

    # Between
    story.append(Paragraph("Between:", tn_r))
    story.append(Spacer(1, 0.3 * inch))

    # Party rows: name centered, role right-aligned in same row
    pn_s = ParagraphStyle('PnS', parent=styles['Normal'],
                          fontName='Times-Bold',  fontSize=12, alignment=TA_CENTER)
    pr_s = ParagraphStyle('PrS', parent=styles['Normal'],
                          fontName='Times-Roman', fontSize=12, alignment=TA_RIGHT)
    col_l = usable_w * 0.72
    col_r = usable_w * 0.28

    applicant = xml_escape(case_info.get('applicant_name', ''))
    respondent = xml_escape(case_info.get('respondent_name', ''))

    if applicant:
        t = Table([[Paragraph(applicant, pn_s), Paragraph("Applicant", pr_s)]],
                  colWidths=[col_l, col_r])
        t.setStyle(TableStyle([('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
                               ('TOPPADDING', (0,0),(-1,-1), 4),
                               ('BOTTOMPADDING', (0,0),(-1,-1), 4)]))
        story.append(t)

    story.append(Paragraph("and", tn_rc))

    if respondent:
        t = Table([[Paragraph(respondent, pn_s), Paragraph("Respondent", pr_s)]],
                  colWidths=[col_l, col_r])
        t.setStyle(TableStyle([('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
                               ('TOPPADDING', (0,0),(-1,-1), 4),
                               ('BOTTOMPADDING', (0,0),(-1,-1), 4)]))
        story.append(t)

    story.append(Spacer(1, 0.9 * inch))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.black))
    story.append(Spacer(1, 0.18 * inch))
    story.append(Paragraph("TENANT&#8217;S EVIDENCE BRIEF", tn_bc))
    story.append(Spacer(1, 0.18 * inch))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.black))
    story.append(PageBreak())

    # ── TABLE OF CONTENTS ─────────────────────────────────────────────────────
    story.append(Paragraph("TABLE OF CONTENTS", tn_bc))
    story.append(Spacer(1, 0.1 * inch))
    story.append(HRFlowable(width="100%", thickness=0.5, color=colors.black))
    story.append(Spacer(1, 0.2 * inch))

    toc_entry_s = ParagraphStyle('TocE', parent=styles['Normal'],
                                 fontName='Times-Roman', fontSize=12, spaceAfter=4)
    toc_hdr_s   = ParagraphStyle('TocH', parent=styles['Normal'],
                                 fontName='Times-Bold',  fontSize=12, spaceAfter=4)
    toc_page_s  = ParagraphStyle('TocP', parent=styles['Normal'],
                                 fontName='Times-Roman', fontSize=12,
                                 spaceAfter=4, alignment=TA_RIGHT)

    toc_data = [[
        Paragraph("Tab",         toc_hdr_s),
        Paragraph("Description", toc_hdr_s),
        Paragraph("Page",        toc_hdr_s),
    ]]
    for i, tab in enumerate(tabs_data, 1):
        pg = str(tab_pages[i]) if (tab_pages and i in tab_pages) else '—'
        toc_data.append([
            Paragraph(str(i),                                toc_entry_s),
            Paragraph(tab.get('title') or ("Tab %d" % i),   toc_entry_s),
            Paragraph(pg,                                    toc_page_s),
        ])

    toc_table = Table(toc_data, colWidths=[0.6 * inch, usable_w - 1.6 * inch, 1 * inch])
    toc_table.setStyle(TableStyle([
        ('TOPPADDING',    (0, 0), (-1, -1), 6),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ('LEFTPADDING',   (0, 0), (-1, -1), 0),
        ('RIGHTPADDING',  (0, 0), (-1, -1), 0),
        ('LINEBELOW',     (0, 0), (-1,  0), 0.5, colors.black),
        ('VALIGN',        (0, 0), (-1, -1), 'MIDDLE'),
        ('ALIGN',         (2, 0), (2,  -1), 'RIGHT'),
    ]))
    story.append(toc_table)
    story.append(PageBreak())

    # ── TAB SECTIONS ──────────────────────────────────────────────────────────
    photo_caption_style = ParagraphStyle(
        'PhotoCaption', parent=styles['Normal'],
        fontName='Times-Roman', fontSize=9,
        textColor=colors.black,
        alignment=TA_CENTER, spaceAfter=16,
    )
    no_file_style = ParagraphStyle(
        'NoFile', parent=styles['Normal'],
        fontName='Times-Roman', fontSize=11,
        textColor=colors.grey, alignment=TA_CENTER,
    )
    doc_body_style = ParagraphStyle(
        'DocBody', parent=styles['Normal'],
        fontName='Times-Roman', fontSize=10,
        leading=14, spaceAfter=4,
    )
    doc_note_style = ParagraphStyle(
        'DocNote', parent=styles['Normal'],
        fontName='Times-Italic', fontSize=9,
        textColor=colors.grey, spaceAfter=8,
    )

    for tab_idx, tab in enumerate(tabs_data, 1):
        story.append(TabDivider(
            tab_number=tab_idx,
            tab_title=tab.get('title', ''),
            width=usable_w,
            height=usable_h,
            page_registry=page_registry,
            registry_key=tab_idx,
        ))
        story.append(PageBreak())

        files = tab.get('images', [])
        if not files:
            story.append(Spacer(1, 2 * inch))
            story.append(Paragraph("No files uploaded for this tab.", no_file_style))
            story.append(PageBreak())
            continue

        img_files = [(i, p) for i, p in enumerate(files) if is_image_file(p)]
        doc_files = [(i, p) for i, p in enumerate(files) if not is_image_file(p)]

        # Embed images — one per page
        img_counter = 0
        for orig_idx, img_path in img_files:
            img_counter += 1
            try:
                rl_img = _rl_image_compressed(
                    img_path,
                    max_w_pts=usable_w,
                    max_h_pts=usable_h - 0.6 * inch,
                )
                story.append(rl_img)
                title_part = (" - " + tab.get('title')) if tab.get('title') else ""
                story.append(Paragraph(
                    "Tab %d%s | Photo %d" % (tab_idx, title_part, img_counter),
                    photo_caption_style
                ))
            except Exception:
                story.append(Paragraph(
                    "[Image could not be loaded: %s]" % os.path.basename(img_path),
                    photo_caption_style
                ))
            story.append(PageBreak())

        # Non-image documents: embed converted pages or fall back to text
        for orig_idx, doc_path in doc_files:
            page_imgs = (doc_page_images or {}).get(doc_path, [])

            if page_imgs:
                # ── Converted document: embed each page directly ──
                for pg_idx, pg_img_path in enumerate(page_imgs):
                    try:
                        rl_img = _rl_image_compressed(
                            pg_img_path,
                            max_w_pts=usable_w,
                            max_h_pts=usable_h,
                        )
                        story.append(rl_img)
                    except Exception:
                        story.append(Paragraph(
                            "[Page could not be rendered]",
                            doc_note_style,
                        ))
                    story.append(PageBreak())
            else:
                # ── Fallback: extract and display full text content ──
                if doc_path not in text_cache:
                    text_cache[doc_path] = extract_text_from_file(doc_path)
                text, ok = text_cache[doc_path]

                if ok and text and text.strip():
                    clean = text.replace('\r\n', '\n').replace('\r', '\n')
                    safe  = xml_escape(clean)
                    html  = safe.replace('\n\n', '<br/><br/>').replace('\n', '<br/>')
                    story.append(Paragraph(html, doc_body_style))
                elif not ok:
                    story.append(Paragraph(
                        "Binary or unsupported format — install PyMuPDF and "
                        "LibreOffice for full document conversion.",
                        doc_note_style,
                    ))
                else:
                    story.append(Paragraph(
                        "[File appears to be empty or contains no extractable text.]",
                        doc_note_style,
                    ))
                story.append(PageBreak())

    return story


# ─── PDF GENERATION (two-pass for accurate TOC page numbers) ──────────────────

def generate_evidence_brief(case_info, tabs_data, output_path, prebuilt_pages=None):
    page_w, page_h = letter
    doc_kwargs = dict(
        pagesize=letter,
        leftMargin=0.75 * inch,
        rightMargin=0.75 * inch,
        topMargin=0.75 * inch,
        bottomMargin=0.75 * inch,
    )
    usable_w = page_w - 1.5 * inch
    usable_h = page_h - 1.5 * inch - 0.5 * inch

    # Pre-convert non-image documents to rendered page images
    # Start with any pre-redacted pages passed in from the frontend
    doc_page_images = dict(prebuilt_pages or {})
    for tab in tabs_data:
        for f in tab.get('images', []):
            if not is_image_file(f) and f not in doc_page_images:
                page_imgs = convert_document_to_images(f)
                if page_imgs:
                    doc_page_images[f] = page_imgs

    # Shared text cache so documents are only read once across both passes
    text_cache = {}

    # Pass 1 — dry run to BytesIO; populates page_registry via TabDivider.draw()
    page_registry = {}
    doc1 = SimpleDocTemplate(io.BytesIO(), **doc_kwargs)
    story1 = _build_story(case_info, tabs_data, usable_w, usable_h,
                          page_registry=page_registry,
                          tab_pages=None,
                          text_cache=text_cache,
                          doc_page_images=doc_page_images)
    doc1.build(story1, canvasmaker=NumberedCanvas)

    # Pass 2 — real render with accurate TOC page numbers
    doc2 = SimpleDocTemplate(output_path, **doc_kwargs)
    story2 = _build_story(case_info, tabs_data, usable_w, usable_h,
                          page_registry=None,
                          tab_pages=page_registry,
                          text_cache=text_cache,
                          doc_page_images=doc_page_images)
    doc2.build(story2, canvasmaker=NumberedCanvas)


# ─── ROUTES ───────────────────────────────────────────────────────────────────

@app.route('/')
def index():
    return render_template('index.html')


def _cleanup_old_uploads():
    """Remove uploaded files older than 1 hour to prevent disk fill."""
    folder = app.config['UPLOAD_FOLDER']
    cutoff = time.time() - 3600
    try:
        for fname in os.listdir(folder):
            fpath = os.path.join(folder, fname)
            if os.path.isfile(fpath) and os.path.getmtime(fpath) < cutoff:
                try:
                    os.remove(fpath)
                except OSError:
                    pass
    except FileNotFoundError:
        pass


@app.route('/upload', methods=['POST'])
def upload_files():
    _cleanup_old_uploads()
    if 'files' not in request.files:
        return jsonify({'error': 'No files provided'}), 400

    saved = []
    for f in request.files.getlist('files'):
        if not f or not f.filename:
            continue
        original_name = f.filename
        ext = original_name.rsplit('.', 1)[-1].lower() if '.' in original_name else 'bin'
        fname = "%s.%s" % (uuid.uuid4().hex, ext)
        path = os.path.join(app.config['UPLOAD_FOLDER'], fname)
        f.save(path)
        # Convert HEIC/HEIF to JPEG so PIL and ReportLab can handle it
        if ext in HEIC_EXTS:
            jpeg_name = '%s.jpg' % uuid.uuid4().hex
            jpeg_path = os.path.join(app.config['UPLOAD_FOLDER'], jpeg_name)
            if _convert_heic_to_jpeg(path, jpeg_path):
                try:
                    os.unlink(path)
                except OSError:
                    pass
                path, fname, ext = jpeg_path, jpeg_name, 'jpg'
        thumb_name = None
        is_image = False
        try:
            with Image.open(path) as img:
                # Fix mobile photo rotation by applying EXIF orientation
                img = ImageOps.exif_transpose(img)
                img.save(path)
                img.thumbnail((300, 300))
                thumb_name = "thumb_%s" % fname
                thumb_path = os.path.join(app.config['UPLOAD_FOLDER'], thumb_name)
                img.save(thumb_path)
                is_image = True
        except Exception:
            pass
        saved.append({'id': fname, 'thumb': thumb_name, 'original': fname,
                      'name': original_name, 'is_image': is_image})

    return jsonify({'files': saved})


@app.route('/thumbnail/<filename>')
def get_thumbnail(filename):
    path = os.path.join(app.config['UPLOAD_FOLDER'], filename)
    if os.path.exists(path):
        return send_file(path)
    return '', 404


# ─── EVIDENCE STATS ────────────────────────────────────────────────────────────

DEFAULT_EVIDENCE_STATS = {
    "N4": {
        "description": "Non-payment of rent",
        "total_cases_analyzed": 0,
        "evidence_types": [
            {"category": "N4 Notice of Termination", "percentage": 92},
            {"category": "Financial records (rent receipts, bank statements, rent ledger)", "percentage": 78},
            {"category": "Lease agreement", "percentage": 65},
            {"category": "Payment history / transaction records", "percentage": 58},
            {"category": "Communication records (emails, text messages, letters)", "percentage": 42},
            {"category": "Legal documents (prior orders, court filings)", "percentage": 28},
            {"category": "Witness testimony", "percentage": 22},
            {"category": "Government/third-party records (inspection reports, municipal notices)", "percentage": 15},
            {"category": "Photos of unit conditions", "percentage": 12},
            {"category": "Maintenance/repair requests or records", "percentage": 8},
        ]
    }
}


@app.route('/evidence-stats')
def evidence_stats():
    stats_path = os.path.join(os.path.dirname(__file__), 'evidence_stats.json')
    if os.path.exists(stats_path):
        try:
            with open(stats_path, 'r') as f:
                return jsonify(json.load(f))
        except (json.JSONDecodeError, IOError):
            pass
    return jsonify(DEFAULT_EVIDENCE_STATS)


@app.route('/preview-pages/<filename>')
def preview_pages(filename):
    """Convert any uploaded file to page images for preview/redaction."""
    safe = secure_filename(filename)
    path = os.path.join(app.config['UPLOAD_FOLDER'], safe)
    if not os.path.exists(path):
        return jsonify({'error': 'File not found'}), 404

    if is_image_file(path):
        thumb = 'thumb_' + safe
        return jsonify({'pages': [safe], 'thumbs': [
            thumb if os.path.exists(os.path.join(app.config['UPLOAD_FOLDER'], thumb)) else safe
        ]})

    page_paths = convert_document_to_images(path)
    if not page_paths:
        return jsonify({'error': 'Could not convert file — PyMuPDF/LibreOffice may not be available'}), 400

    page_ids, thumb_ids = [], []
    for pg_path in page_paths:
        pg_name = os.path.basename(pg_path)
        page_ids.append(pg_name)
        thumb_name = 'thumb_' + pg_name
        thumb_path = os.path.join(app.config['UPLOAD_FOLDER'], thumb_name)
        if not os.path.exists(thumb_path):
            try:
                with Image.open(pg_path) as img:
                    img.thumbnail((300, 300))
                    img.save(thumb_path)
            except Exception:
                pass
        thumb_ids.append(thumb_name)

    return jsonify({'pages': page_ids, 'thumbs': thumb_ids})


@app.route('/redact', methods=['POST'])
def redact_file():
    """Receive a redacted image blob, overwrite the original, regenerate thumbnail."""
    file_id = request.form.get('file_id')
    blob = request.files.get('image')
    if not file_id or not blob:
        return jsonify({'error': 'Missing data'}), 400

    # Validate file_id to prevent path traversal
    safe_name = secure_filename(file_id)
    if safe_name != file_id or '/' in file_id or '\\' in file_id:
        return jsonify({'error': 'Invalid file id'}), 400

    original_path = os.path.join(app.config['UPLOAD_FOLDER'], safe_name)
    if not os.path.exists(original_path):
        return jsonify({'error': 'File not found'}), 404

    # Overwrite the original with the redacted version
    blob.save(original_path)

    # Regenerate thumbnail
    thumb_name = 'thumb_%s' % safe_name
    thumb_path = os.path.join(app.config['UPLOAD_FOLDER'], thumb_name)
    try:
        with Image.open(original_path) as img:
            img = ImageOps.exif_transpose(img)
            img.thumbnail((300, 300))
            img.save(thumb_path)
    except Exception:
        pass

    return jsonify({'thumb': thumb_name})


@app.route('/generate', methods=['POST'])
def generate():
    data = request.get_json()
    if not data:
        return jsonify({'error': 'No data provided'}), 400

    case_info = data.get('case_info', {})
    tabs_data = data.get('tabs', [])
    preview_pages_map = data.get('preview_pages', {})  # {orig_id: [page_id, ...]}

    for tab in tabs_data:
        resolved = []
        for img_id in tab.get('images', []):
            path = os.path.join(app.config['UPLOAD_FOLDER'], img_id)
            if os.path.exists(path):
                resolved.append(path)
        tab['images'] = resolved

    # Build pre-generated page image mapping from redacted pages
    prebuilt = {}
    for orig_id, page_ids in preview_pages_map.items():
        orig_path = os.path.join(app.config['UPLOAD_FOLDER'], secure_filename(orig_id))
        pages = [os.path.join(app.config['UPLOAD_FOLDER'], secure_filename(pid))
                 for pid in page_ids
                 if os.path.exists(os.path.join(app.config['UPLOAD_FOLDER'], secure_filename(pid)))]
        if pages:
            prebuilt[orig_path] = pages

    output_path = os.path.join(
        app.config['UPLOAD_FOLDER'], "brief_%s.pdf" % uuid.uuid4().hex
    )

    try:
        generate_evidence_brief(case_info, tabs_data, output_path, prebuilt_pages=prebuilt)
        download_name = "Evidence_Brief_%s.pdf" % case_info.get('file_number', 'LTB')
        return send_file(
            output_path,
            mimetype='application/pdf',
            as_attachment=True,
            download_name=download_name,
        )
    except Exception as e:
        logging.exception("PDF generation failed")
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)
    debug = os.environ.get('FLASK_DEBUG', 'false').lower() == 'true'
    port = int(os.environ.get('PORT', 5050))
    app.run(debug=debug, host='0.0.0.0', port=port)


## Step 8: Start the App

Launches the Flask server on port **5050** and provides the URL to open the tool.

- In **Google Colab**: uses the built-in proxy to generate a public HTTPS link you can open in a new tab
- In **local Jupyter**: opens at `http://localhost:5050`

The server runs in a background thread so this cell returns immediately. If the port is already in use, it will be freed first.

In [ ]:
import threading, socket, time

# Kill any existing Flask server on port 5050
def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if _port_in_use(5050):
    import subprocess
    subprocess.run("fuser -k 5050/tcp", shell=True, capture_output=True)
    time.sleep(1)

# Start Flask in a background thread
threading.Thread(
    target=lambda: app.run(port=5050, use_reloader=False),
    daemon=True
).start()

time.sleep(2)

# Try Colab proxy (works inside Colab browser)
try:
    from google.colab.output import eval_js
    url = eval_js("google.colab.kernel.proxyPort(5050)")
    print(f"\n{'='*60}")
    print(f"  App is running!")
    print(f"  Open: {url}")
    print(f"{'='*60}\n")
except Exception:
    try:
        from pyngrok import ngrok
        public_url = ngrok.connect(5050)
        print(f"\n{'='*60}")
        print(f"  App is running!")
        print(f"  Open: {public_url}")
        print(f"{'='*60}\n")
    except Exception:
        print("\n" + "="*60)
        print("  App is running on http://localhost:5050")
        print("  (Install pyngrok for a public URL)")
        print("="*60 + "\n")
